# Base analysis

In [1]:
"""Base analysis framework for all RQ analyzers."""

from abc import ABC, abstractmethod
from pathlib import Path
import json
import warnings
from datetime import datetime
from dataclasses import dataclass, field
import typing as t

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns

from ..utils.data_loaders import EvaluationDataLoader, create_filter

@dataclass
class AnalysisConfig:
    """Base configuration for analysis modules."""
    results_dir: Path
    output_dir: Path
    analysis_name: str
    
    # Statistical parameters
    confidence_level: float = 0.95
    significance_threshold: float = 0.05
    n_bootstrap: int = 10000
    
    # Visualization parameters
    figure_format: str = "png"
    figure_dpi: int = 300
    figure_size: tuple[int, int] = (10, 8)
    style: str = "whitegrid"
    
    # Analysis parameters
    min_samples_per_condition: int = 3
    max_missing_data_fraction: float = 0.2
    
    def validate(self) -> bool:
        """Validate analysis configuration."""
        if not self.results_dir.exists():
            raise FileNotFoundError(f"Results directory not found: {self.results_dir}")
        
        if self.confidence_level <= 0 or self.confidence_level >= 1:
            raise ValueError("Confidence level must be between 0 and 1")
        
        if self.significance_threshold <= 0 or self.significance_threshold >= 1:
            raise ValueError("Significance threshold must be between 0 and 1")
        
        return True

class BaseAnalyzer(ABC):
    """Base class for all research question analyzers."""
    
    def __init__(self, config: AnalysisConfig):
        """Initialize base analyzer."""
        self.config = config
        self.config.validate()
        
        self.data_loader = EvaluationDataLoader(config.results_dir)
        self.setup_output_directories()
        self.setup_visualization()
        
        # Load common data
        self.manifest = self.data_loader.load_experiment_manifest()
        self.model_registry = self.data_loader.load_model_registry()
        
    def setup_output_directories(self) -> None:
        """Create necessary output directories."""
        directories = [
            self.config.output_dir,
            self.config.output_dir / "figures",
            self.config.output_dir / "data",
            self.config.output_dir / "reports",
        ]
        
        for directory in directories:
            directory.mkdir(parents=True, exist_ok=True)
    
    def setup_visualization(self) -> None:
        """Setup matplotlib and seaborn styling."""
        plt.style.use('default')
        sns.set_style(self.config.style)
        sns.set_palette("husl")
        
        # Set default figure parameters
        plt.rcParams['figure.figsize'] = self.config.figure_size
        plt.rcParams['figure.dpi'] = self.config.figure_dpi
        plt.rcParams['font.size'] = 12
        plt.rcParams['axes.titlesize'] = 14
        plt.rcParams['axes.labelsize'] = 12
        plt.rcParams['xtick.labelsize'] = 10
        plt.rcParams['ytick.labelsize'] = 10
        plt.rcParams['legend.fontsize'] = 11
    
    @abstractmethod
    def run_analysis(self) -> dict[str, t.Any]:
        """Run the specific analysis for this research question."""
        pass
    
    @abstractmethod
    def generate_report(self, results: dict[str, t.Any]) -> Path:
        """Generate analysis report."""
        pass
    
    def save_results(self, results: dict[str, t.Any], filename: str = "results.json") -> Path:
        """Save analysis results to JSON file."""
        output_path = self.config.output_dir / "data" / filename
        
        # Make results JSON serializable
        serializable_results = self._make_json_serializable(results)
        
        with open(output_path, "w") as f:
            json.dump(serializable_results, f, indent=2, default=str)
        
        return output_path
    
    def save_figure(
        self, 
        fig: plt.Figure, 
        filename: str, 
        tight_layout: bool = True
    ) -> Path:
        """Save matplotlib figure."""
        if tight_layout:
            fig.tight_layout()
        
        output_path = self.config.output_dir / "figures" / f"{filename}.{self.config.figure_format}"
        fig.savefig(output_path, dpi=self.config.figure_dpi, bbox_inches='tight')
        plt.close(fig)
        
        return output_path
    
    def bootstrap_confidence_interval(
        self, 
        data: list[float], 
        statistic_func: t.Callable = np.mean
    ) -> tuple[float, float]:
        """Compute bootstrap confidence interval for a statistic."""
        if len(data) < 2:
            return float('nan'), float('nan')
        
        bootstrap_stats = []
        for _ in range(self.config.n_bootstrap):
            sample = np.random.choice(data, size=len(data), replace=True)
            bootstrap_stats.append(statistic_func(sample))
        
        alpha = 1 - self.config.confidence_level
        lower = np.percentile(bootstrap_stats, 100 * alpha / 2)
        upper = np.percentile(bootstrap_stats, 100 * (1 - alpha / 2))
        
        return float(lower), float(upper)
    
    def fit_curve(
        self, 
        x: np.ndarray, 
        y: np.ndarray, 
        curve_type: str = "logistic"
    ) -> dict[str, t.Any]:
        """Fit a curve to data points."""
        if len(x) < 3 or len(y) < 3:
            return {"success": False, "error": "Insufficient data points"}
        
        try:
            if curve_type == "logistic":
                return self._fit_logistic_curve(x, y)
            elif curve_type == "exponential":
                return self._fit_exponential_curve(x, y)
            elif curve_type == "power":
                return self._fit_power_curve(x, y)
            elif curve_type == "polynomial":
                return self._fit_polynomial_curve(x, y)
            else:
                raise ValueError(f"Unknown curve type: {curve_type}")
                
        except Exception as e:
            return {"success": False, "error": str(e)}
    
    def _fit_logistic_curve(self, x: np.ndarray, y: np.ndarray) -> dict[str, t.Any]:
        """Fit logistic curve: y = L / (1 + exp(-k*(x-x0)))."""
        from scipy.optimize import curve_fit
        
        def logistic(x, L, k, x0):
            return L / (1 + np.exp(-k * (x - x0)))
        
        # Initial parameter guesses
        L_init = np.max(y)
        k_init = 1.0
        x0_init = np.median(x)
        
        try:
            params, covariance = curve_fit(
                logistic, x, y, 
                p0=[L_init, k_init, x0_init],
                maxfev=5000
            )
            
            L, k, x0 = params
            y_pred = logistic(x, L, k, x0)
            r2 = r2_score(y, y_pred)
            
            # Calculate parameter uncertainties
            param_errors = np.sqrt(np.diag(covariance))
            
            return {
                "success": True,
                "curve_type": "logistic",
                "parameters": {"L": L, "k": k, "x0": x0},
                "parameter_errors": {"L": param_errors[0], "k": param_errors[1], "x0": param_errors[2]},
                "r2": r2,
                "predictions": y_pred.tolist(),
                "equation": f"y = {L:.3f} / (1 + exp(-{k:.3f} * (x - {x0:.3f})))"
            }
            
        except Exception as e:
            return {"success": False, "error": f"Logistic fit failed: {e}"}
    
    def _fit_exponential_curve(self, x: np.ndarray, y: np.ndarray) -> dict[str, t.Any]:
        """Fit exponential curve: y = a * exp(b * x)."""
        from scipy.optimize import curve_fit
        
        def exponential(x, a, b):
            return a * np.exp(b * x)
        
        try:
            # Use log transform for initial guess
            log_y = np.log(np.maximum(y, 1e-10))
            poly_fit = np.polyfit(x, log_y, 1)
            a_init = np.exp(poly_fit[1])
            b_init = poly_fit[0]
            
            params, covariance = curve_fit(
                exponential, x, y,
                p0=[a_init, b_init],
                maxfev=5000
            )
            
            a, b = params
            y_pred = exponential(x, a, b)
            r2 = r2_score(y, y_pred)
            
            param_errors = np.sqrt(np.diag(covariance))
            
            return {
                "success": True,
                "curve_type": "exponential",
                "parameters": {"a": a, "b": b},
                "parameter_errors": {"a": param_errors[0], "b": param_errors[1]},
                "r2": r2,
                "predictions": y_pred.tolist(),
                "equation": f"y = {a:.3f} * exp({b:.3f} * x)"
            }
            
        except Exception as e:
            return {"success": False, "error": f"Exponential fit failed: {e}"}
    
    def _fit_power_curve(self, x: np.ndarray, y: np.ndarray) -> dict[str, t.Any]:
        """Fit power curve: y = a * x^b."""
        from scipy.optimize import curve_fit
        
        def power(x, a, b):
            return a * np.power(x, b)
        
        try:
            # Filter out zero or negative values for log transform
            mask = (x > 0) & (y > 0)
            if np.sum(mask) < 3:
                return {"success": False, "error": "Insufficient positive data points"}
            
            x_pos, y_pos = x[mask], y[mask]
            
            # Use log transform for initial guess
            log_x, log_y = np.log(x_pos), np.log(y_pos)
            poly_fit = np.polyfit(log_x, log_y, 1)
            a_init = np.exp(poly_fit[1])
            b_init = poly_fit[0]
            
            params, covariance = curve_fit(
                power, x_pos, y_pos,
                p0=[a_init, b_init],
                maxfev=5000
            )
            
            a, b = params
            y_pred = power(x, a, b)
            r2 = r2_score(y[mask], power(x_pos, a, b))
            
            param_errors = np.sqrt(np.diag(covariance))
            
            return {
                "success": True,
                "curve_type": "power",
                "parameters": {"a": a, "b": b},
                "parameter_errors": {"a": param_errors[0], "b": param_errors[1]},
                "r2": r2,
                "predictions": y_pred.tolist(),
                "equation": f"y = {a:.3f} * x^{b:.3f}"
            }
            
        except Exception as e:
            return {"success": False, "error": f"Power fit failed: {e}"}
    
    def _fit_polynomial_curve(self, x: np.ndarray, y: np.ndarray, degree: int = 2) -> dict[str, t.Any]:
        """Fit polynomial curve."""
        try:
            coeffs = np.polyfit(x, y, degree)
            y_pred = np.polyval(coeffs, x)
            r2 = r2_score(y, y_pred)
            
            # Create equation string
            terms = []
            for i, coeff in enumerate(coeffs):
                power = degree - i
                if power == 0:
                    terms.append(f"{coeff:.3f}")
                elif power == 1:
                    terms.append(f"{coeff:.3f}*x")
                else:
                    terms.append(f"{coeff:.3f}*x^{power}")
            
            equation = "y = " + " + ".join(terms)
            
            return {
                "success": True,
                "curve_type": f"polynomial_degree_{degree}",
                "parameters": {"coefficients": coeffs.tolist(), "degree": degree},
                "r2": r2,
                "predictions": y_pred.tolist(),
                "equation": equation
            }
            
        except Exception as e:
            return {"success": False, "error": f"Polynomial fit failed: {e}"}
    
    def statistical_test(
        self, 
        group1: list[float], 
        group2: list[float], 
        test_type: str = "ttest"
    ) -> dict[str, t.Any]:
        """Perform statistical test between two groups."""
        if len(group1) < 2 or len(group2) < 2:
            return {"success": False, "error": "Insufficient data for statistical test"}
        
        try:
            if test_type == "ttest":
                statistic, p_value = stats.ttest_ind(group1, group2)
                test_name = "Independent t-test"
            elif test_type == "mannwhitney":
                statistic, p_value = stats.mannwhitneyu(group1, group2, alternative='two-sided')
                test_name = "Mann-Whitney U test"
            elif test_type == "ks":
                statistic, p_value = stats.ks_2samp(group1, group2)
                test_name = "Kolmogorov-Smirnov test"
            else:
                raise ValueError(f"Unknown test type: {test_type}")
            
            effect_size = self._calculate_effect_size(group1, group2)
            significant = p_value < self.config.significance_threshold
            
            return {
                "success": True,
                "test_name": test_name,
                "statistic": float(statistic),
                "p_value": float(p_value),
                "significant": significant,
                "effect_size": effect_size,
                "group1_stats": {"mean": np.mean(group1), "std": np.std(group1), "n": len(group1)},
                "group2_stats": {"mean": np.mean(group2), "std": np.std(group2), "n": len(group2)}
            }
            
        except Exception as e:
            return {"success": False, "error": f"Statistical test failed: {e}"}
    
    def _calculate_effect_size(self, group1: list[float], group2: list[float]) -> float:
        """Calculate Cohen's d effect size."""
        mean1, mean2 = np.mean(group1), np.mean(group2)
        std1, std2 = np.std(group1, ddof=1), np.std(group2, ddof=1)
        n1, n2 = len(group1), len(group2)
        
        # Pooled standard deviation
        pooled_std = np.sqrt(((n1 - 1) * std1**2 + (n2 - 1) * std2**2) / (n1 + n2 - 2))
        
        if pooled_std == 0:
            return 0.0
        
        return (mean1 - mean2) / pooled_std
    
    def _make_json_serializable(self, obj: t.Any) -> t.Any:
        """Convert numpy arrays and other non-serializable objects to JSON-compatible format."""
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, (np.integer, np.floating)):
            return obj.item()
        elif isinstance(obj, np.bool_):
            return bool(obj)
        elif isinstance(obj, dict):
            return {key: self._make_json_serializable(value) for key, value in obj.items()}
        elif isinstance(obj, (list, tuple)):
            return [self._make_json_serializable(item) for item in obj]
        elif isinstance(obj, Path):
            return str(obj)
        elif pd.isna(obj):
            return None
        else:
            return obj
    
    def create_summary_statistics(self, data: pd.DataFrame, group_by: list[str]) -> pd.DataFrame:
        """Create summary statistics grouped by specified columns."""
        numeric_cols = data.select_dtypes(include=[np.number]).columns
        
        summary = data.groupby(group_by)[numeric_cols].agg([
            'count', 'mean', 'std', 'min', 'max', 'median'
        ]).round(4)
        
        return summary
    
    def validate_data_completeness(self, data: pd.DataFrame, required_columns: list[str]) -> bool:
        """Validate that data has sufficient completeness for analysis."""
        # Check required columns exist
        missing_columns = [col for col in required_columns if col not in data.columns]
        if missing_columns:
            warnings.warn(f"Missing required columns: {missing_columns}")
            return False
        
        # Check for excessive missing data
        for col in required_columns:
            missing_fraction = data[col].isna().mean()
            if missing_fraction > self.config.max_missing_data_fraction:
                warnings.warn(f"Column {col} has {missing_fraction:.1%} missing data")
                return False
        
        # Check minimum sample size
        if len(data) < self.config.min_samples_per_condition:
            warnings.warn(f"Insufficient data: {len(data)} samples < {self.config.min_samples_per_condition}")
            return False
        
        return True

ImportError: attempted relative import with no known parent package

# RQ1: Emergence Analysis

In [ ]:
"""RQ1: Analyze when ICL capabilities emerge as training task diversity increases."""

from pathlib import Path
from dataclasses import dataclass, field
import typing as t

import pandas as pd
import numpy as np


from ..utils.data_loaders import create_filter

@dataclass
class EmergenceConfig(AnalysisConfig):
    """Configuration for emergence analysis."""
    # Emergence detection parameters
    emergence_threshold: float = 0.6  # Accuracy threshold for emergence
    min_context_size: int = 1
    max_context_size: int = 8
    
    # Analysis parameters
    diversity_levels: list[int] = field(default_factory=lambda: [8, 16, 32, 64, 128])
    smoothing_window: int = 3  # For smoothing emergence curves
    
    # Curve fitting parameters
    fit_curve_types: list[str] = field(default_factory=lambda: ["logistic", "exponential"])

class EmergenceAnalyzer(BaseAnalyzer):
    """Analyzer for ICL emergence patterns (RQ1)."""
    
    def __init__(self, config: EmergenceConfig):
        """Initialize emergence analyzer."""
        super().__init__(config)
        self.config: EmergenceConfig = config
    
    def run_analysis(self) -> dict[str, t.Any]:
        """Run comprehensive emergence analysis."""
        print("Starting RQ1: ICL Emergence Analysis")
        print("=" * 50)
        
        # Load and validate data
        emergence_data = self._load_emergence_data()
        if not self.validate_data_completeness(emergence_data, 
                                             ["config_L", "config_m", "n_train", "context_size", "accuracy"]):
            raise ValueError("Insufficient data for emergence analysis")
        
        print(f"Loaded {len(emergence_data)} evaluation records")
        print(f"Configurations: {emergence_data[['config_L', 'config_m']].drop_duplicates().shape[0]}")
        print(f"Diversity levels: {sorted(emergence_data['n_train'].unique())}")
        
        results = {}
        
        # 1. Detect emergence thresholds for each configuration
        print("\n1. Detecting emergence thresholds...")
        emergence_thresholds = self._detect_emergence_thresholds(emergence_data)
        results["emergence_thresholds"] = emergence_thresholds
        
        # 2. Analyze emergence curves
        print("2. Analyzing emergence curves...")
        emergence_curves = self._analyze_emergence_curves(emergence_data)
        results["emergence_curves"] = emergence_curves
        
        # 3. Fit mathematical models to emergence patterns
        print("3. Fitting emergence models...")
        emergence_models = self._fit_emergence_models(emergence_data)
        results["emergence_models"] = emergence_models
        
        # 4. Statistical analysis of emergence patterns
        print("4. Statistical analysis...")
        statistical_analysis = self._statistical_emergence_analysis(emergence_data, emergence_thresholds)
        results["statistical_analysis"] = statistical_analysis
        
        # 5. Generate visualizations
        print("5. Generating visualizations...")
        self._generate_emergence_visualizations(emergence_data, results)
        
        # Save results
        self.save_results(results, "rq1_emergence_results.json")
        
        print("\nRQ1 Analysis completed successfully!")
        return results
    
    def _load_emergence_data(self) -> pd.DataFrame:
        """Load data for emergence analysis."""
        # Load within-config ICL performance data
        filter_dict = (create_filter()
                      .transfer_condition("within_config")
                      .control_type("normal")
                      .build())
        
        data = self.data_loader.load_icl_performance(filters=filter_dict)
        
        # Add model metadata
        data = data.merge(
            self.model_registry[["model_id", "config_L", "config_m", "n_train", "checkpoint_step", "model_type"]],
            on="model_id",
            how="left"
        )
        
        # Filter by diversity levels if specified
        if self.config.diversity_levels:
            data = data[data["n_train"].isin(self.config.diversity_levels)]
        
        return data
    
    def _detect_emergence_thresholds(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Detect N_train thresholds where ICL emerges for each configuration."""
        emergence_results = {}
        
        # Group by configuration
        for (config_L, config_m), config_group in data.groupby(["config_L", "config_m"]):
            config_key = f"L{config_L}_m{config_m}"
            print(f"  Analyzing {config_key}...")
            
            # For each diversity level, compute best ICL performance
            diversity_performance = {}
            
            for n_train, diversity_group in config_group.groupby("n_train"):
                # Get best performance across context sizes for this diversity level
                best_accuracy = diversity_group.groupby("context_size")["accuracy"].mean().max()
                diversity_performance[n_train] = best_accuracy
            
            # Find emergence threshold
            emergence_n_train = None
            for n_train in sorted(diversity_performance.keys()):
                if diversity_performance[n_train] >= self.config.emergence_threshold:
                    emergence_n_train = n_train
                    break
            
            # Calculate emergence statistics
            diversity_levels = sorted(diversity_performance.keys())
            performance_values = [diversity_performance[n] for n in diversity_levels]
            
            emergence_results[config_key] = {
                "config_L": config_L,
                "config_m": config_m,
                "emergence_threshold": self.config.emergence_threshold,
                "emergence_n_train": emergence_n_train,
                "diversity_performance": diversity_performance,
                "max_performance": max(performance_values) if performance_values else 0.0,
                "final_performance": performance_values[-1] if performance_values else 0.0,
                "performance_gain": (max(performance_values) - min(performance_values)) if len(performance_values) > 1 else 0.0,
            }
        
        return emergence_results
    
    def _analyze_emergence_curves(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze detailed emergence curves for each configuration."""
        curve_results = {}
        
        # Group by configuration and context size
        for (config_L, config_m), config_group in data.groupby(["config_L", "config_m"]):
            config_key = f"L{config_L}_m{config_m}"
            
            config_curves = {}
            
            # Analyze emergence curve for each context size
            for context_size, context_group in config_group.groupby("context_size"):
                if context_size < self.config.min_context_size or context_size > self.config.max_context_size:
                    continue
                
                # Compute mean accuracy for each diversity level
                diversity_accuracy = context_group.groupby("n_train")["accuracy"].agg([
                    "mean", "std", "count"
                ]).reset_index()
                
                # Calculate confidence intervals
                confidence_intervals = []
                for _, row in diversity_accuracy.iterrows():
                    if row["count"] > 1:
                        lower, upper = self.bootstrap_confidence_interval(
                            context_group[context_group["n_train"] == row["n_train"]]["accuracy"].tolist()
                        )
                    else:
                        lower, upper = row["mean"], row["mean"]
                    
                    confidence_intervals.append({"lower": lower, "upper": upper})
                
                diversity_accuracy["ci_lower"] = [ci["lower"] for ci in confidence_intervals]
                diversity_accuracy["ci_upper"] = [ci["upper"] for ci in confidence_intervals]
                
                config_curves[f"k{context_size}"] = {
                    "context_size": context_size,
                    "emergence_curve": diversity_accuracy.to_dict("records"),
                    "peak_performance": diversity_accuracy["mean"].max(),
                    "final_performance": diversity_accuracy["mean"].iloc[-1] if len(diversity_accuracy) > 0 else 0.0,
                }
            
            curve_results[config_key] = {
                "config_L": config_L,
                "config_m": config_m,
                "curves_by_context": config_curves,
            }
        
        return curve_results
    
    def _fit_emergence_models(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Fit mathematical models to emergence patterns."""
        model_results = {}
        
        for (config_L, config_m), config_group in data.groupby(["config_L", "config_m"]):
            config_key = f"L{config_L}_m{config_m}"
            
            # Get aggregate performance across all context sizes
            diversity_performance = config_group.groupby("n_train")["accuracy"].mean()
            
            if len(diversity_performance) < 3:
                continue
            
            x = np.array(sorted(diversity_performance.index))
            y = np.array([diversity_performance[n] for n in x])
            
            # Fit different curve types
            fitted_models = {}
            for curve_type in self.config.fit_curve_types:
                fit_result = self.fit_curve(x, y, curve_type)
                fitted_models[curve_type] = fit_result
            
            # Select best model based on R²
            best_model = None
            best_r2 = -1
            for curve_type, model in fitted_models.items():
                if model.get("success", False) and model.get("r2", -1) > best_r2:
                    best_model = curve_type
                    best_r2 = model["r2"]
            
            model_results[config_key] = {
                "config_L": config_L,
                "config_m": config_m,
                "fitted_models": fitted_models,
                "best_model": best_model,
                "best_r2": best_r2,
                "data_points": {"x": x.tolist(), "y": y.tolist()},
            }
        
        return model_results
    
    def _statistical_emergence_analysis(
        self, 
        data: pd.DataFrame, 
        emergence_thresholds: dict[str, t.Any]
    ) -> dict[str, t.Any]:
        """Statistical analysis of emergence patterns."""
        stats_results = {}
        
        # Extract emergence N_train values
        emergence_values = []
        complexity_values = []
        
        for config_key, result in emergence_thresholds.items():
            if result["emergence_n_train"] is not None:
                emergence_values.append(result["emergence_n_train"])
                # Use L + m as complexity measure
                complexity_values.append(result["config_L"] + result["config_m"])
        
        if len(emergence_values) < 2:
            return {"error": "Insufficient emergence data for statistical analysis"}
        
        # Correlation between complexity and emergence threshold
        if len(complexity_values) == len(emergence_values):
            correlation = np.corrcoef(complexity_values, emergence_values)[0, 1]
            
            # Statistical test for correlation
            from scipy.stats import pearsonr
            corr_stat, corr_p = pearsonr(complexity_values, emergence_values)
        else:
            correlation = np.nan
            corr_stat, corr_p = np.nan, np.nan
        
        # Compare emergence patterns across configurations
        config_comparisons = {}
        configs = list(emergence_thresholds.keys())
        
        for i, config1 in enumerate(configs):
            for config2 in configs[i+1:]:
                result1 = emergence_thresholds[config1]
                result2 = emergence_thresholds[config2]
                
                if result1["

# RQ2: Scaling Law analyis

In [ ]:
"""RQ2: Analyze context scaling laws and optimal context length patterns."""

from pathlib import Path
from dataclasses import dataclass, field
import typing as t

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize_scalar

from .base_analyzer import BaseAnalyzer, AnalysisConfig
from ..utils.data_loaders import create_filter


@dataclass
class ScalingConfig(AnalysisConfig):
    """Configuration for scaling analysis."""
    # Scaling law parameters
    min_context_size: int = 1
    max_context_size: int = 16
    scaling_law_types: list[str] = field(default_factory=lambda: ["power", "logarithmic", "exponential"])
    
    # k* optimization parameters
    performance_weight: float = 1.0
    efficiency_weight: float = 0.1
    cost_function: str = "accuracy_per_token"  # "accuracy_per_token" or "custom"
    
    # Analysis parameters
    min_diversity_for_analysis: int = 16
    convergence_threshold: float = 0.01


class ScalingAnalyzer(BaseAnalyzer):
    """Analyzer for context scaling laws and optimization (RQ2)."""
    
    def __init__(self, config: ScalingConfig):
        """Initialize scaling analyzer."""
        super().__init__(config)
        self.config: ScalingConfig = config
    
    def run_analysis(self) -> dict[str, t.Any]:
        """Run comprehensive scaling analysis."""
        print("Starting RQ2: Context Scaling Analysis")
        print("=" * 50)
        
        # Load and validate data
        scaling_data = self._load_scaling_data()
        if not self.validate_data_completeness(scaling_data, 
                                             ["config_L", "config_m", "n_train", "context_size", "accuracy"]):
            raise ValueError("Insufficient data for scaling analysis")
        
        print(f"Loaded {len(scaling_data)} evaluation records")
        print(f"Context sizes: {sorted(scaling_data['context_size'].unique())}")
        
        results = {}
        
        # 1. Fit scaling laws for each configuration/diversity combination
        print("\n1. Fitting scaling laws...")
        scaling_laws = self._fit_scaling_laws(scaling_data)
        results["scaling_laws"] = scaling_laws
        
        # 2. Find optimal context lengths (k*)
        print("2. Computing optimal context lengths...")
        optimal_contexts = self._compute_optimal_contexts(scaling_data)
        results["optimal_contexts"] = optimal_contexts
        
        # 3. Analyze scaling patterns across configurations
        print("3. Analyzing scaling patterns...")
        scaling_patterns = self._analyze_scaling_patterns(scaling_laws)
        results["scaling_patterns"] = scaling_patterns
        
        # 4. Statistical analysis of scaling behavior
        print("4. Statistical analysis...")
        statistical_analysis = self._statistical_scaling_analysis(scaling_data, scaling_laws)
        results["statistical_analysis"] = statistical_analysis
        
        # 5. Generate visualizations
        print("5. Generating visualizations...")
        self._generate_scaling_visualizations(scaling_data, results)
        
        # Save results
        self.save_results(results, "rq2_scaling_results.json")
        
        print("\nRQ2 Analysis completed successfully!")
        return results
    
    def _load_scaling_data(self) -> pd.DataFrame:
        """Load data for scaling analysis."""
        # Load within-config ICL performance data
        filter_dict = (create_filter()
                      .transfer_condition("within_config")
                      .control_type("normal")
                      .build())
        
        data = self.data_loader.load_icl_performance(filters=filter_dict)
        
        # Add model metadata
        data = data.merge(
            self.model_registry[["model_id", "config_L", "config_m", "n_train", "checkpoint_step"]],
            on="model_id",
            how="left"
        )
        
        # Filter by context size range
        data = data[
            (data["context_size"] >= self.config.min_context_size) &
            (data["context_size"] <= self.config.max_context_size)
        ]
        
        # Only analyze configurations with sufficient diversity
        data = data[data["n_train"] >= self.config.min_diversity_for_analysis]
        
        return data
    
    def _fit_scaling_laws(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Fit scaling laws for each configuration and diversity level."""
        scaling_results = {}
        
        # Group by configuration and diversity level
        for (config_L, config_m, n_train), group in data.groupby(["config_L", "config_m", "n_train"]):
            group_key = f"L{config_L}_m{config_m}_n{n_train}"
            print(f"  Fitting scaling laws for {group_key}...")
            
            # Compute mean accuracy for each context size
            context_performance = group.groupby("context_size")["accuracy"].agg([
                "mean", "std", "count"
            ]).reset_index()
            
            if len(context_performance) < 3:
                continue
            
            x = context_performance["context_size"].values
            y = context_performance["mean"].values
            y_std = context_performance["std"].fillna(0).values
            
            # Fit different scaling law types
            fitted_laws = {}
            for law_type in self.config.scaling_law_types:
                fit_result = self._fit_specific_scaling_law(x, y, y_std, law_type)
                fitted_laws[law_type] = fit_result
            
            # Select best scaling law
            best_law = self._select_best_scaling_law(fitted_laws)
            
            scaling_results[group_key] = {
                "config_L": config_L,
                "config_m": config_m,
                "n_train": n_train,
                "context_performance": context_performance.to_dict("records"),
                "fitted_laws": fitted_laws,
                "best_law": best_law,
                "data_quality": {
                    "n_context_sizes": len(context_performance),
                    "context_range": [int(x.min()), int(x.max())],
                    "mean_std": float(y_std.mean())
                }
            }
        
        return scaling_results
    
    def _fit_specific_scaling_law(
        self, 
        x: np.ndarray, 
        y: np.ndarray, 
        y_std: np.ndarray, 
        law_type: str
    ) -> dict[str, t.Any]:
        """Fit a specific type of scaling law."""
        try:
            if law_type == "power":
                return self._fit_power_scaling(x, y, y_std)
            elif law_type == "logarithmic":
                return self._fit_logarithmic_scaling(x, y, y_std)
            elif law_type == "exponential":
                return self._fit_exponential_scaling(x, y, y_std)
            else:
                return {"success": False, "error": f"Unknown scaling law type: {law_type}"}
        except Exception as e:
            return {"success": False, "error": str(e)}
    
    def _fit_power_scaling(self, x: np.ndarray, y: np.ndarray, y_std: np.ndarray) -> dict[str, t.Any]:
        """Fit power law: y = a * x^b + c."""
        from scipy.optimize import curve_fit
        
        def power_law(x, a, b, c):
            return a * np.power(x, b) + c
        
        # Initial parameter guesses
        a_init = (y[-1] - y[0]) / (x[-1]**0.5 - x[0]**0.5) if len(y) > 1 else 1.0
        b_init = 0.5
        c_init = y.min()
        
        try:
            # Use standard deviation as weights if available
            sigma = np.maximum(y_std, 0.001)  # Avoid zero weights
            
            params, covariance = curve_fit(
                power_law, x, y, 
                p0=[a_init, b_init, c_init],
                sigma=sigma,
                maxfev=5000
            )
            
            a, b, c = params
            y_pred = power_law(x, a, b, c)
            
            # Calculate metrics
            r2 = self._calculate_r2(y, y_pred)
            aic = self._calculate_aic(y, y_pred, len(params))
            
            param_errors = np.sqrt(np.diag(covariance))
            
            return {
                "success": True,
                "law_type": "power",
                "parameters": {"a": a, "b": b, "c": c},
                "parameter_errors": {"a": param_errors[0], "b": param_errors[1], "c": param_errors[2]},
                "r2": r2,
                "aic": aic,
                "predictions": y_pred.tolist(),
                "equation": f"y = {a:.3f} * x^{b:.3f} + {c:.3f}"
            }
            
        except Exception as e:
            return {"success": False, "error": f"Power law fit failed: {e}"}
    
    def _fit_logarithmic_scaling(self, x: np.ndarray, y: np.ndarray, y_std: np.ndarray) -> dict[str, t.Any]:
        """Fit logarithmic law: y = a * log(x + b) + c."""
        from scipy.optimize import curve_fit
        
        def log_law(x, a, b, c):
            return a * np.log(x + b) + c
        
        # Initial parameter guesses
        a_init = (y[-1] - y[0]) / (np.log(x[-1] + 1) - np.log(x[0] + 1)) if len(y) > 1 else 1.0
        b_init = 1.0
        c_init = y.min()
        
        try:
            sigma = np.maximum(y_std, 0.001)
            
            params, covariance = curve_fit(
                log_law, x, y,
                p0=[a_init, b_init, c_init],
                sigma=sigma,
                maxfev=5000
            )
            
            a, b, c = params
            y_pred = log_law(x, a, b, c)
            
            r2 = self._calculate_r2(y, y_pred)
            aic = self._calculate_aic(y, y_pred, len(params))
            
            param_errors = np.sqrt(np.diag(covariance))
            
            return {
                "success": True,
                "law_type": "logarithmic",
                "parameters": {"a": a, "b": b, "c": c},
                "parameter_errors": {"a": param_errors[0], "b": param_errors[1], "c": param_errors[2]},
                "r2": r2,
                "aic": aic,
                "predictions": y_pred.tolist(),
                "equation": f"y = {a:.3f} * log(x + {b:.3f}) + {c:.3f}"
            }
            
        except Exception as e:
            return {"success": False, "error": f"Logarithmic law fit failed: {e}"}
    
    def _fit_exponential_scaling(self, x: np.ndarray, y: np.ndarray, y_std: np.ndarray) -> dict[str, t.Any]:
        """Fit exponential law: y = a * (1 - exp(-b * x)) + c."""
        from scipy.optimize import curve_fit
        
        def exp_law(x, a, b, c):
            return a * (1 - np.exp(-b * x)) + c
        
        # Initial parameter guesses
        a_init = y.max() - y.min()
        b_init = 1.0 / np.mean(x)
        c_init = y.min()
        
        try:
            sigma = np.maximum(y_std, 0.001)
            
            params, covariance = curve_fit(
                exp_law, x, y,
                p0=[a_init, b_init, c_init],
                sigma=sigma,
                maxfev=5000
            )
            
            a, b, c = params
            y_pred = exp_law(x, a, b, c)
            
            r2 = self._calculate_r2(y, y_pred)
            aic = self._calculate_aic(y, y_pred, len(params))
            
            param_errors = np.sqrt(np.diag(covariance))
            
            return {
                "success": True,
                "law_type": "exponential",
                "parameters": {"a": a, "b": b, "c": c},
                "parameter_errors": {"a": param_errors[0], "b": param_errors[1], "c": param_errors[2]},
                "r2": r2,
                "aic": aic,
                "predictions": y_pred.tolist(),
                "equation": f"y = {a:.3f} * (1 - exp(-{b:.3f} * x)) + {c:.3f}"
            }
            
        except Exception as e:
            return {"success": False, "error": f"Exponential law fit failed: {e}"}
    
    def _select_best_scaling_law(self, fitted_laws: dict[str, t.Any]) -> dict[str, t.Any]:
        """Select the best scaling law based on AIC or R²."""
        successful_laws = {k: v for k, v in fitted_laws.items() if v.get("success", False)}
        
        if not successful_laws:
            return {"best_law_type": None, "reason": "No successful fits"}
        
        # Select based on AIC (lower is better)
        best_law_type = min(successful_laws.keys(), 
                           key=lambda k: successful_laws[k].get("aic", float('inf')))
        
        best_law = successful_laws[best_law_type]
        
        return {
            "best_law_type": best_law_type,
            "best_law_data": best_law,
            "comparison": {k: {"r2": v.get("r2", 0), "aic": v.get("aic", float('inf'))} 
                          for k, v in successful_laws.items()}
        }
    
    def _compute_optimal_contexts(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Compute optimal context lengths for each configuration."""
        optimal_results = {}
        
        # Group by configuration and diversity level
        for (config_L, config_m, n_train), group in data.groupby(["config_L", "config_m", "n_train"]):
            group_key = f"L{config_L}_m{config_m}_n{n_train}"
            
            # Compute performance vs efficiency trade-off
            context_metrics = group.groupby("context_size").agg({
                "accuracy": ["mean", "std"],
            }).round(4)
            
            context_metrics.columns = ["accuracy_mean", "accuracy_std"]
            context_metrics = context_metrics.reset_index()
            
            # Compute efficiency metric (accuracy per token)
            if self.config.cost_function == "accuracy_per_token":
                context_metrics["efficiency"] = context_metrics["accuracy_mean"] / context_metrics["context_size"]
            
            # Find optimal context using different criteria
            optimal_accuracy_k = context_metrics.loc[context_metrics["accuracy_mean"].idxmax(), "context_size"]
            optimal_efficiency_k = context_metrics.loc[context_metrics["efficiency"].idxmax(), "context_size"]
            
            # Composite optimization
            context_metrics["composite_score"] = (
                self.config.performance_weight * context_metrics["accuracy_mean"] +
                self.config.efficiency_weight * context_metrics["efficiency"]
            )
            optimal_composite_k = context_metrics.loc[context_metrics["composite_score"].idxmax(), "context_size"]
            
            # Find convergence point (where performance plateaus)
            convergence_k = self._find_convergence_point(context_metrics)
            
            optimal_results[group_key] = {
                "config_L": config_L,
                "config_m": config_m,
                "n_train": n_train,
                "optimal_contexts": {
                    "accuracy_optimized": int(optimal_accuracy_k),
                    "efficiency_optimized": int(optimal_efficiency_k),
                    "composite_optimized": int(optimal_composite_k),
                    "convergence_point": convergence_k
                },
                "context_metrics": context_metrics.to_dict("records"),
                "max_accuracy": float(context_metrics["accuracy_mean"].max()),
                "max_efficiency": float(context_metrics["efficiency"].max())
            }
        
        return optimal_results
    
    def _find_convergence_point(self, context_metrics: pd.DataFrame) -> int | None:
        """Find the context size where performance converges (plateaus)."""
        if len(context_metrics) < 3:
            return None
        
        accuracies = context_metrics["accuracy_mean"].values
        context_sizes = context_metrics["context_size"].values
        
        # Calculate differences between consecutive points
        diffs = np.diff(accuracies)
        
        # Find first point where improvement is below threshold
        for i, diff in enumerate(diffs):
            if diff < self.config.convergence_threshold:
                return int(context_sizes[i + 1])
        
        # If no convergence found, return the maximum context size tested
        return int(context_sizes[-1])
    
    def _analyze_scaling_patterns(self, scaling_laws: dict[str, t.Any]) -> dict[str, t.Any]:
        """Analyze patterns across all scaling laws."""
        patterns = {
            "law_type_distribution": {},
            "parameter_patterns": {},
            "complexity_relationships": {}
        }
        
        # Count distribution of best law types
        law_types = []
        complexities = []
        parameters_by_type = {}
        
        for group_key, result in scaling_laws.items():
            best_law = result.get("best_law", {})
            law_type = best_law.get("best_law_type")
            
            if law_type:
                law_types.append(law_type)
                complexities.append(result["config_L"] + result["config_m"])
                
                # Collect parameters for pattern analysis
                if law_type not in parameters_by_type:
                    parameters_by_type[law_type] = []
                
                best_law_data = best_law.get("best_law_data", {})
                if best_law_data.get("success", False):
                    parameters_by_type[law_type].append(best_law_data.get("parameters", {}))
        
        # Distribution of law types
        for law_type in set(law_types):
            patterns["law_type_distribution"][law_type] = law_types.count(law_type)
        
        # Parameter patterns for each law type
        for law_type, param_list in parameters_by_type.items():
            if param_list:
                param_stats = {}
                param_names = param_list[0].keys() if param_list else []
                
                for param_name in param_names:
                    values = [p.get(param_name) for p in param_list if p.get(param_name) is not None]
                    if values:
                        param_stats[param_name] = {
                            "mean": float(np.mean(values)),
                            "std": float(np.std(values)),
                            "min": float(np.min(values)),
                            "max": float(np.max(values))
                        }
                
                patterns["parameter_patterns"][law_type] = param_stats
        
        # Complexity vs scaling relationships
        if complexities and law_types:
            patterns["complexity_relationships"] = {
                "complexity_range": [min(complexities), max(complexities)],
                "law_type_by_complexity": list(zip(complexities, law_types))
            }
        
        return patterns
    
    def _statistical_scaling_analysis(
        self, 
        data: pd.DataFrame, 
        scaling_laws: dict[str, t.Any]
    ) -> dict[str, t.Any]:
        """Statistical analysis of scaling behavior."""
        stats_results = {}
        
        # Collect R² values for comparison
        r2_values = []
        law_types = []
        
        for result in scaling_laws.values():
            best_law = result.get("best_law", {})
            best_law_data = best_law.get("best_law_data", {})
            if best_law_data.get("success", False):
                r2_values.append(best_law_data.get("r2", 0))
                law_types.append(best_law.get("best_law_type", "unknown"))
        
        if r2_values:
            stats_results["fit_quality"] = {
                "mean_r2": float(np.mean(r2_values)),
                "std_r2": float(np.std(r2_values)),
                "min_r2": float(np.min(r2_values)),
                "max_r2": float(np.max(r2_values)),
                "n_fits": len(r2_values)
            }
        
        # Compare scaling behavior across law types
        law_type_r2 = {}
        for law_type in set(law_types):
            type_r2 = [r2 for r2, lt in zip(r2_values, law_types) if lt == law_type]
            if type_r2:
                law_type_r2[law_type] = {
                    "mean_r2": float(np.mean(type_r2)),
                    "std_r2": float(np.std(type_r2)),
                    "count": len(type_r2)
                }
        
        stats_results["law_type_comparison"] = law_type_r2
        
        return stats_results
    
    def _generate_scaling_visualizations(self, data: pd.DataFrame, results: dict[str, t.Any]) -> None:
        """Generate comprehensive scaling visualizations."""
        # 1. Scaling curves by configuration
        self._plot_scaling_curves(data, results["scaling_laws"])
        
        # 2. Optimal context distributions
        self._plot_optimal_context_distributions(results["optimal_contexts"])
        
        # 3. Scaling law comparison
        self._plot_scaling_law_comparison(results["scaling_laws"])
        
        # 4. Parameter relationships
        self._plot_parameter_relationships(results["scaling_patterns"])
    
    def _plot_scaling_curves(self, data: pd.DataFrame, scaling_laws: dict[str, t.Any]) -> None:
        """Plot scaling curves for different configurations."""
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        axes = axes.ravel()
        
        configs = list(scaling_laws.keys())[:4]  # Show first 4 configurations
        
        for i, config_key in enumerate(configs):
            ax = axes[i]
            result = scaling_laws[config_key]
            
            # Plot observed data points
            context_perf = pd.DataFrame(result["context_performance"])
            ax.errorbar(context_perf["context_size"], context_perf["mean"], 
                       yerr=context_perf["std"], fmt='o', label="Observed", alpha=0.7)
            
            # Plot best fit curve
            best_law = result.get("best_law", {})
            best_law_data = best_law.get("best_law_data", {})
            if best_law_data.get("success", False):
                predictions = best_law_data.get("predictions", [])
                if predictions:
                    ax.plot(context_perf["context_size"], predictions, 
                           '--', label=f"Best fit ({best_law.get('best_law_type', 'unknown')})")
            
            ax.set_xlabel("Context Size (k)")
            ax.set_ylabel("Accuracy")
            ax.set_title(f"Scaling Curve: {config_key}")
            ax.legend()
            ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        self.save_figure(fig, "scaling_curves_by_config")
    
    def _plot_optimal_context_distributions(self, optimal_contexts: dict[str, t.Any]) -> None:
        """Plot distributions of optimal context lengths."""
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Extract optimal context values
        criteria = ["accuracy_optimized", "efficiency_optimized", "composite_optimized", "convergence_point"]
        titles = ["Accuracy Optimized", "Efficiency Optimized", "Composite Optimized", "Convergence Point"]
        
        for i, (criterion, title) in enumerate(zip(criteria, titles)):
            ax = axes[i // 2, i % 2]
            
            values = []
            for result in optimal_contexts.values():
                optimal_k = result["optimal_contexts"].get(criterion)
                if optimal_k is not None:
                    values.append(optimal_k)
            
            if values:
                ax.hist(values, bins=range(min(values), max(values) + 2), alpha=0.7, edgecolor='black')
                ax.set_xlabel("Context Size (k*)")
                ax.set_ylabel("Frequency")
                ax.set_title(f"Distribution of {title}")
                ax.grid(True, alpha=0.3)
                
                # Add statistics
                ax.axvline(np.mean(values), color='red', linestyle='--', label=f'Mean: {np.mean(values):.1f}')
                ax.legend()
        
        plt.tight_layout()
        self.save_figure(fig, "optimal_context_distributions")
    
    def _plot_scaling_law_comparison(self, scaling_laws: dict[str, t.Any]) -> None:
        """Compare different scaling law types."""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Count law types
        law_type_counts = {}
        r2_by_law_type = {}
        
        for result in scaling_laws.values():
            best_law = result.get("best_law", {})
            law_type = best_law.get("best_law_type")
            best_law_data = best_law.get("best_law_data", {})
            
            if law_type and best_law_data.get("success", False):
                law_type_counts[law_type] = law_type_counts.get(law_type, 0) + 1
                
                if law_type not in r2_by_law_type:
                    r2_by_law_type[law_type] = []
                r2_by_law_type[law_type].append(best_law_data.get("r2", 0))
        
        # Plot law type distribution
        if law_type_counts:
            ax1.bar(law_type_counts.keys(), law_type_counts.values())
            ax1.set_xlabel("Scaling Law Type")
            ax1.set_ylabel("Frequency")
            ax1.set_title("Distribution of Best Scaling Law Types")
            ax1.tick_params(axis='x', rotation=45)
        
        # Plot R² comparison by law type
        if r2_by_law_type:
            law_types = list(r2_by_law_type.keys())
            r2_data = [r2_by_law_type[lt] for lt in law_types]
            
            ax2.boxplot(r2_data, labels=law_types)
            ax2.set_xlabel("Scaling Law Type")
            ax2.set_ylabel("R² Score")
            ax2.set_title("Fit Quality by Scaling Law Type")
            ax2.tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        self.save_figure(fig, "scaling_law_comparison")
    
    def _plot_parameter_relationships(self, scaling_patterns: dict[str, t.Any]) -> None:
        """Plot relationships between scaling parameters and complexity."""
        parameter_patterns = scaling_patterns.get("parameter_patterns", {})
        
        if not parameter_patterns:
            return
        
        n_law_types = len(parameter_patterns)
        if n_law_types == 0:
            return
        
        fig, axes = plt.subplots(1, min(n_law_types, 3), figsize=(5 * min(n_law_types, 3), 5))
        if n_law_types == 1:
            axes = [axes]
        
        for i, (law_type, param_stats) in enumerate(list(parameter_patterns.items())[:3]):
            ax = axes[i] if len(axes) > 1 else axes[0]
            
            param_names = list(param_stats.keys())
            means = [param_stats[name]["mean"] for name in param_names]
            stds = [param_stats[name]["std"] for name in param_names]
            
            ax.bar(param_names, means, yerr=stds, capsize=5, alpha=0.7)
            ax.set_xlabel("Parameter")
            ax.set_ylabel("Value")
            ax.set_title(f"Parameters for {law_type.title()} Law")
            ax.tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        self.save_figure(fig, "parameter_relationships")
    
    def _calculate_r2(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        """Calculate R² score."""
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
        return 1 - (ss_res / ss_tot) if ss_tot != 0 else 0.0
    
    def _calculate_aic(self, y_true: np.ndarray, y_pred: np.ndarray, n_params: int) -> float:
        """Calculate Akaike Information Criterion."""
        n = len(y_true)
        mse = np.mean((y_true - y_pred) ** 2)
        return n * np.log(mse) + 2 * n_params
    
    def generate_report(self, results: dict[str, t.Any]) -> Path:
        """Generate scaling analysis report."""
        report_lines = [
            "# RQ2: Context Scaling Analysis Report",
            "",
            f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            "",
            "## Executive Summary",
            "",
            "This report analyzes context scaling laws and optimal context length patterns",
            "across different model configurations and training diversity levels.",
            "",
            "## Key Findings",
            "",
        ]
        
        # Scaling laws summary
        scaling_laws = results.get("scaling_laws", {})
        if scaling_laws:
            report_lines.extend([
                f"### Scaling Laws Analysis",
                f"- Total configurations analyzed: {len(scaling_laws)}",
                "",
            ])
            
            # Best law type distribution
            scaling_patterns = results.get("scaling_patterns", {})
            law_distribution = scaling_patterns.get("law_type_distribution", {})
            if law_distribution:
                report_lines.append("**Scaling Law Type Distribution:**")
                for law_type, count in sorted(law_distribution.items(), key=lambda x: x[1], reverse=True):
                    percentage = (count / sum(law_distribution.values())) * 100
                    report_lines.append(f"- {law_type.title()}: {count} ({percentage:.1f}%)")
                report_lines.append("")
        
        # Optimal contexts summary
        optimal_contexts = results.get("optimal_contexts", {})
        if optimal_contexts:
            accuracy_opts = []
            efficiency_opts = []
            composite_opts = []
            
            for result in optimal_contexts.values():
                opts = result["optimal_contexts"]
                if opts.get("accuracy_optimized") is not None:
                    accuracy_opts.append(opts["accuracy_optimized"])
                if opts.get("efficiency_optimized") is not None:
                    efficiency_opts.append(opts["efficiency_optimized"])
                if opts.get("composite_optimized") is not None:
                    composite_opts.append(opts["composite_optimized"])
            
            report_lines.extend([
                "### Optimal Context Lengths (k*)",
                "",
                f"**Accuracy-Optimized k*:**",
                f"- Mean: {np.mean(accuracy_opts):.1f}, Std: {np.std(accuracy_opts):.1f}" if accuracy_opts else "- No data available",
                f"- Range: [{min(accuracy_opts)} - {max(accuracy_opts)}]" if accuracy_opts else "",
                "",
                f"**Efficiency-Optimized k*:**",
                f"- Mean: {np.mean(efficiency_opts):.1f}, Std: {np.std(efficiency_opts):.1f}" if efficiency_opts else "- No data available",
                f"- Range: [{min(efficiency_opts)} - {max(efficiency_opts)}]" if efficiency_opts else "",
                "",
                f"**Composite-Optimized k*:**",
                f"- Mean: {np.mean(composite_opts):.1f}, Std: {np.std(composite_opts):.1f}" if composite_opts else "- No data available",
                f"- Range: [{min(composite_opts)} - {max(composite_opts)}]" if composite_opts else "",
                "",
            ])
        
        # Statistical analysis summary
        statistical_analysis = results.get("statistical_analysis", {})
        fit_quality = statistical_analysis.get("fit_quality", {})
        if fit_quality:
            report_lines.extend([
                "### Statistical Analysis",
                "",
                "**Overall Fit Quality:**",
                f"- Mean R²: {fit_quality.get('mean_r2', 0):.3f}",
                f"- Std R²: {fit_quality.get('std_r2', 0):.3f}",
                f"- Best R²: {fit_quality.get('max_r2', 0):.3f}",
                f"- Worst R²: {fit_quality.get('min_r2', 0):.3f}",
                f"- Number of successful fits: {fit_quality.get('n_fits', 0)}",
                "",
            ])
        
        # Law type comparison
        law_type_comparison = statistical_analysis.get("law_type_comparison", {})
        if law_type_comparison:
            report_lines.extend([
                "**Fit Quality by Law Type:**",
                "",
            ])
            for law_type, stats in sorted(law_type_comparison.items(), 
                                        key=lambda x: x[1].get("mean_r2", 0), reverse=True):
                report_lines.extend([
                    f"- **{law_type.title()}:**",
                    f"  - Mean R²: {stats.get('mean_r2', 0):.3f}",
                    f"  - Std R²: {stats.get('std_r2', 0):.3f}",
                    f"  - Count: {stats.get('count', 0)}",
                ])
            report_lines.append("")
        
        # Parameter patterns
        parameter_patterns = scaling_patterns.get("parameter_patterns", {})
        if parameter_patterns:
            report_lines.extend([
                "### Parameter Patterns",
                "",
            ])
            for law_type, param_stats in parameter_patterns.items():
                report_lines.extend([
                    f"**{law_type.title()} Law Parameters:**",
                    "",
                ])
                for param_name, stats in param_stats.items():
                    report_lines.extend([
                        f"- {param_name}:",
                        f"  - Mean: {stats.get('mean', 0):.3f} ± {stats.get('std', 0):.3f}",
                        f"  - Range: [{stats.get('min', 0):.3f}, {stats.get('max', 0):.3f}]",
                    ])
                report_lines.append("")
        
        # Detailed configuration analysis
        report_lines.extend([
            "## Detailed Analysis by Configuration",
            "",
        ])
        
        for config_key, result in list(scaling_laws.items())[:5]:  # Show first 5 configs
            config_L = result["config_L"]
            config_m = result["config_m"]
            n_train = result["n_train"]
            
            report_lines.extend([
                f"### Configuration: L={config_L}, m={config_m}, n_train={n_train}",
                "",
            ])
            
            # Best scaling law
            best_law = result.get("best_law", {})
            best_law_data = best_law.get("best_law_data", {})
            if best_law_data.get("success", False):
                law_type = best_law.get("best_law_type", "unknown")
                r2 = best_law_data.get("r2", 0)
                equation = best_law_data.get("equation", "N/A")
                
                report_lines.extend([
                    f"**Best Scaling Law:** {law_type.title()}",
                    f"- R² Score: {r2:.3f}",
                    f"- Equation: {equation}",
                    "",
                ])
            
            # Data quality metrics
            data_quality = result.get("data_quality", {})
            if data_quality:
                report_lines.extend([
                    "**Data Quality:**",
                    f"- Context sizes tested: {data_quality.get('n_context_sizes', 0)}",
                    f"- Context range: {data_quality.get('context_range', [0, 0])}",
                    f"- Mean standard deviation: {data_quality.get('mean_std', 0):.3f}",
                    "",
                ])
            
            # Optimal context for this config
            config_optimal = optimal_contexts.get(config_key, {})
            if config_optimal:
                opts = config_optimal["optimal_contexts"]
                report_lines.extend([
                    "**Optimal Context Lengths:**",
                    f"- Accuracy-optimized: k* = {opts.get('accuracy_optimized', 'N/A')}",
                    f"- Efficiency-optimized: k* = {opts.get('efficiency_optimized', 'N/A')}",
                    f"- Composite-optimized: k* = {opts.get('composite_optimized', 'N/A')}",
                    f"- Convergence point: k* = {opts.get('convergence_point', 'N/A')}",
                    "",
                    f"**Performance Metrics:**",
                    f"- Maximum accuracy: {config_optimal.get('max_accuracy', 0):.3f}",
                    f"- Maximum efficiency: {config_optimal.get('max_efficiency', 0):.3f}",
                    "",
                ])
        
        # Methodology
        report_lines.extend([
            "## Methodology",
            "",
            "### Scaling Law Fitting",
            "",
            "Three types of scaling laws were fitted to the data:",
            "",
            "1. **Power Law:** y = a × x^b + c",
            "2. **Logarithmic Law:** y = a × log(x + b) + c", 
            "3. **Exponential Law:** y = a × (1 - exp(-b × x)) + c",
            "",
            "The best law for each configuration was selected based on the Akaike Information Criterion (AIC).",
            "",
            "### Optimal Context Length Computation",
            "",
            "Four optimization criteria were used:",
            "",
            "1. **Accuracy-Optimized:** Context length maximizing absolute accuracy",
            "2. **Efficiency-Optimized:** Context length maximizing accuracy per token",
            "3. **Composite-Optimized:** Weighted combination of accuracy and efficiency",
            f"4. **Convergence Point:** Context length where improvement drops below {self.config.convergence_threshold}",
            "",
            f"Composite optimization used weights: performance={self.config.performance_weight}, efficiency={self.config.efficiency_weight}",
            "",
            "### Statistical Analysis",
            "",
            "- Bootstrap confidence intervals (95%) were computed for all metrics",
            f"- Minimum {self.config.min_samples_per_condition} samples required per condition",
            f"- Maximum {self.config.max_missing_data_fraction*100:.1f}% missing data allowed",
            "",
            "## Conclusions",
            "",
            "This analysis provides insights into:",
            "- Which mathematical forms best describe context scaling behavior",
            "- Optimal context lengths for different optimization objectives",
            "- How scaling patterns vary across model configurations",
            "- The relationship between model complexity and scaling behavior",
            "",
            f"For detailed visualizations, see the figures directory: {self.config.output_dir / 'figures'}",
            "",
        ])
        
        # Write report
        report_path = self.config.output_dir / "reports" / "rq2_scaling_report.md"
        report_path.write_text("\n".join(report_lines))
        
        print(f"Report generated: {report_path}")
        return report_path

# RQ3: Attention Pattern

In [ ]:
"""RQ3: Analyze attention patterns and internal representations for ICL mechanistic understanding."""

from pathlib import Path
from dataclasses import dataclass, field
import typing as t
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from .base_analyzer import BaseAnalyzer, AnalysisConfig
from ..utils.data_loaders import create_filter


@dataclass
class MechanisticConfig(AnalysisConfig):
    """Configuration for mechanistic analysis."""
    # Attention analysis parameters
    attention_layers: list[int] | None = None  # None for all layers
    attention_heads: list[int] | None = None   # None for all heads
    context_positions: list[int] | None = None # None for all positions
    
    # Representation analysis parameters
    probe_hidden_dims: list[int] = field(default_factory=lambda: [64, 128])
    probe_regularization: float = 0.01
    probe_max_iter: int = 1000
    
    # Specialization analysis parameters
    specialization_metric: str = "entropy"  # "entropy", "variance", "gini"
    min_attention_threshold: float = 0.01
    
    # PCA parameters
    pca_components: int = 10
    pca_variance_threshold: float = 0.95
    
    # Clustering parameters
    n_clusters: int = 5
    clustering_method: str = "kmeans"  # "kmeans", "hierarchical"


class MechanisticAnalyzer(BaseAnalyzer):
    """Analyzer for mechanistic understanding of ICL (RQ3)."""
    
    def __init__(self, config: MechanisticConfig):
        """Initialize mechanistic analyzer."""
        super().__init__(config)
        self.config: MechanisticConfig = config
        
    def run_analysis(self) -> dict[str, t.Any]:
        """Run comprehensive mechanistic analysis."""
        print("Starting RQ3: Mechanistic Analysis")
        print("=" * 50)
        
        # Load and validate data
        attention_data = self._load_attention_data()
        representation_data = self._load_representation_data()
        performance_data = self._load_performance_data()
        
        print(f"Loaded attention data for {len(attention_data)} evaluations")
        print(f"Loaded representation data for {len(representation_data)} evaluations")
        print(f"Loaded performance data for {len(performance_data)} evaluations")
        
        results = {}
        
        # 1. Attention pattern analysis
        print("\n1. Analyzing attention patterns...")
        attention_analysis = self._analyze_attention_patterns(attention_data, performance_data)
        results["attention_analysis"] = attention_analysis
        
        # 2. Layer specialization analysis
        print("2. Analyzing layer specialization...")
        specialization_analysis = self._analyze_layer_specialization(attention_data)
        results["specialization_analysis"] = specialization_analysis
        
        # 3. Representation analysis
        print("3. Analyzing internal representations...")
        representation_analysis = self._analyze_representations(representation_data, performance_data)
        results["representation_analysis"] = representation_analysis
        
        # 4. Probe training for interpretability
        print("4. Training interpretability probes...")
        probe_analysis = self._train_interpretability_probes(representation_data, performance_data)
        results["probe_analysis"] = probe_analysis
        
        # 5. Mechanistic pattern correlation with performance
        print("5. Correlating patterns with performance...")
        correlation_analysis = self._correlate_patterns_with_performance(results, performance_data)
        results["correlation_analysis"] = correlation_analysis
        
        # 6. Generate visualizations
        print("6. Generating visualizations...")
        self._generate_mechanistic_visualizations(results)
        
        # Save results
        self.save_results(results, "rq3_mechanistic_results.json")
        
        print("\nRQ3 Analysis completed successfully!")
        return results
    
    def _load_attention_data(self) -> pd.DataFrame:
        """Load attention pattern data."""
        # Load attention data from the attention_data directory
        attention_dir = self.config.results_dir / "raw_evaluations" / "attention_data"
        
        if not attention_dir.exists():
            print("Warning: No attention data directory found")
            return pd.DataFrame()
        
        # Load attention metadata
        metadata_path = attention_dir / "metadata.json"
        if metadata_path.exists():
            import json
            with open(metadata_path) as f:
                attention_metadata = json.load(f)
        else:
            attention_metadata = {}
        
        # Load attention patterns for analysis
        attention_records = []
        
        model_dirs = [d for d in attention_dir.iterdir() if d.is_dir() and d.name != "by_model"]
        if (attention_dir / "by_model").exists():
            model_dirs = list((attention_dir / "by_model").iterdir())
        
        for model_dir in model_dirs[:10]:  # Limit to first 10 models for memory
            model_id = model_dir.name
            
            for attention_file in model_dir.glob("*.npz"):
                try:
                    # Load attention weights
                    attention_data = np.load(attention_file)
                    
                    # Extract metadata from filename (assumes format: modelid_configL_configm_ntrain_k_seqid.npz)
                    filename_parts = attention_file.stem.split("_")
                    if len(filename_parts) >= 6:
                        config_L = int(filename_parts[1])
                        config_m = int(filename_parts[2])
                        n_train = int(filename_parts[3])
                        context_size = int(filename_parts[4])
                        sequence_id = int(filename_parts[5])
                        
                        # Extract attention weights (shape: [layers, heads, seq_len, seq_len])
                        attention_weights = attention_data.get("attention_weights")
                        if attention_weights is not None:
                            attention_records.append({
                                "model_id": model_id,
                                "config_L": config_L,
                                "config_m": config_m,
                                "n_train": n_train,
                                "context_size": context_size,
                                "sequence_id": sequence_id,
                                "attention_weights": attention_weights,
                                "n_layers": attention_weights.shape[0],
                                "n_heads": attention_weights.shape[1],
                                "seq_length": attention_weights.shape[2]
                            })
                
                except Exception as e:
                    print(f"Warning: Could not load attention file {attention_file}: {e}")
                    continue
        
        return pd.DataFrame(attention_records)
    
    def _load_representation_data(self) -> pd.DataFrame:
        """Load internal representation data."""
        # Load representation data from the representations directory
        repr_dir = self.config.results_dir / "raw_evaluations" / "representations"
        
        if not repr_dir.exists():
            print("Warning: No representation data directory found")
            return pd.DataFrame()
        
        # Load representation metadata
        metadata_path = repr_dir / "metadata.json"
        if metadata_path.exists():
            import json
            with open(metadata_path) as f:
                repr_metadata = json.load(f)
        else:
            repr_metadata = {}
        
        # Load representation data for analysis
        repr_records = []
        
        layer_dirs = [d for d in repr_dir.iterdir() if d.is_dir() and d.name.startswith("layer_")]
        
        for layer_dir in layer_dirs:
            layer_num = int(layer_dir.name.split("_")[1])
            
            for repr_file in layer_dir.glob("*.npz")[:50]:  # Limit files for memory
                try:
                    # Load hidden states
                    repr_data = np.load(repr_file)
                    
                    # Extract metadata from filename
                    filename_parts = repr_file.stem.split("_")
                    if len(filename_parts) >= 6:
                        model_id = filename_parts[0]
                        config_L = int(filename_parts[1])
                        config_m = int(filename_parts[2])
                        n_train = int(filename_parts[3])
                        context_size = int(filename_parts[4])
                        sequence_id = int(filename_parts[5])
                        
                        # Extract hidden states (shape: [seq_len, hidden_dim])
                        hidden_states = repr_data.get("hidden_states")
                        if hidden_states is not None:
                            repr_records.append({
                                "model_id": model_id,
                                "config_L": config_L,
                                "config_m": config_m,
                                "n_train": n_train,
                                "context_size": context_size,
                                "sequence_id": sequence_id,
                                "layer": layer_num,
                                "hidden_states": hidden_states,
                                "seq_length": hidden_states.shape[0],
                                "hidden_dim": hidden_states.shape[1]
                            })
                
                except Exception as e:
                    print(f"Warning: Could not load representation file {repr_file}: {e}")
                    continue
        
        return pd.DataFrame(repr_records)
    
    def _load_performance_data(self) -> pd.DataFrame:
        """Load performance data for correlation analysis."""
        # Load ICL performance data
        filter_dict = (create_filter()
                      .transfer_condition("within_config")
                      .control_type("normal")
                      .build())
        
        data = self.data_loader.load_icl_performance(filters=filter_dict)
        
        # Add model metadata
        data = data.merge(
            self.model_registry[["model_id", "config_L", "config_m", "n_train", "checkpoint_step"]],
            on="model_id",
            how="left"
        )
        
        return data
    
    def _analyze_attention_patterns(self, attention_data: pd.DataFrame, performance_data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze attention patterns and their relationship to ICL performance."""
        if attention_data.empty:
            return {"error": "No attention data available"}
        
        pattern_results = {}
        
        # Analyze attention patterns by configuration
        for (config_L, config_m, n_train), group in attention_data.groupby(["config_L", "config_m", "n_train"]):
            group_key = f"L{config_L}_m{config_m}_n{n_train}"
            
            attention_metrics = []
            
            for _, row in group.iterrows():
                attention_weights = row["attention_weights"]
                
                # Compute various attention metrics
                metrics = self._compute_attention_metrics(attention_weights)
                metrics.update({
                    "model_id": row["model_id"],
                    "context_size": row["context_size"],
                    "sequence_id": row["sequence_id"]
                })
                attention_metrics.append(metrics)
            
            if attention_metrics:
                pattern_results[group_key] = {
                    "config_L": config_L,
                    "config_m": config_m,
                    "n_train": n_train,
                    "attention_metrics": attention_metrics,
                    "n_samples": len(attention_metrics)
                }
        
        # Aggregate attention patterns
        aggregated_patterns = self._aggregate_attention_patterns(pattern_results)
        pattern_results["aggregated_patterns"] = aggregated_patterns
        
        return pattern_results
    
    def _compute_attention_metrics(self, attention_weights: np.ndarray) -> dict[str, float]:
        """Compute various metrics from attention weights."""
        n_layers, n_heads, seq_len, _ = attention_weights.shape
        
        metrics = {}
        
        # 1. Attention entropy (measure of attention spread)
        entropies = []
        for layer in range(n_layers):
            for head in range(n_heads):
                attn = attention_weights[layer, head]
                for pos in range(seq_len):
                    attn_dist = attn[pos] + 1e-8  # Avoid log(0)
                    entropy = -np.sum(attn_dist * np.log(attn_dist))
                    entropies.append(entropy)
        
        metrics["mean_attention_entropy"] = float(np.mean(entropies))
        metrics["std_attention_entropy"] = float(np.std(entropies))
        
        # 2. Attention concentration (inverse of entropy)
        metrics["mean_attention_concentration"] = 1.0 / (1.0 + metrics["mean_attention_entropy"])
        
        # 3. Layer-wise attention variance
        layer_variances = []
        for layer in range(n_layers):
            layer_attn = attention_weights[layer].mean(axis=0)  # Average over heads
            layer_variances.append(float(np.var(layer_attn)))
        
        metrics["attention_layer_variance"] = layer_variances
        metrics["mean_layer_variance"] = float(np.mean(layer_variances))
        
        # 4. Head specialization (variance across heads within layers)
        head_specializations = []
        for layer in range(n_layers):
            head_patterns = []
            for head in range(n_heads):
                head_pattern = attention_weights[layer, head].flatten()
                head_patterns.append(head_pattern)
            
            if len(head_patterns) > 1:
                head_matrix = np.stack(head_patterns)
                specialization = float(np.mean(np.var(head_matrix, axis=0)))
                head_specializations.append(specialization)
        
        metrics["head_specialization"] = head_specializations
        metrics["mean_head_specialization"] = float(np.mean(head_specializations)) if head_specializations else 0.0
        
        # 5. Position-wise attention patterns
        position_attentions = []
        for pos in range(seq_len):
            pos_attn = attention_weights[:, :, pos, :].mean(axis=(0, 1))  # Average over layers and heads
            position_attentions.append(pos_attn.tolist())
        
        metrics["position_attention_patterns"] = position_attentions
        
        # 6. Attention to previous tokens vs. future tokens
        if seq_len > 1:
            prev_attention = []
            for layer in range(n_layers):
                for head in range(n_heads):
                    for pos in range(1, seq_len):
                        prev_attn = np.sum(attention_weights[layer, head, pos, :pos])
                        prev_attention.append(prev_attn)
            
            metrics["mean_previous_attention"] = float(np.mean(prev_attention))
            metrics["std_previous_attention"] = float(np.std(prev_attention))
        
        return metrics
    
    def _aggregate_attention_patterns(self, pattern_results: dict[str, t.Any]) -> dict[str, t.Any]:
        """Aggregate attention patterns across configurations."""
        aggregated = {
            "entropy_patterns": {},
            "specialization_patterns": {},
            "position_patterns": {}
        }
        
        for group_key, result in pattern_results.items():
            if "attention_metrics" not in result:
                continue
                
            metrics_list = result["attention_metrics"]
            
            # Aggregate entropy metrics
            entropies = [m["mean_attention_entropy"] for m in metrics_list]
            concentrations = [m["mean_attention_concentration"] for m in metrics_list]
            
            aggregated["entropy_patterns"][group_key] = {
                "mean_entropy": float(np.mean(entropies)),
                "std_entropy": float(np.std(entropies)),
                "mean_concentration": float(np.mean(concentrations)),
                "std_concentration": float(np.std(concentrations))
            }
            
            # Aggregate specialization metrics
            specializations = [m["mean_head_specialization"] for m in metrics_list]
            layer_variances = [m["mean_layer_variance"] for m in metrics_list]
            
            aggregated["specialization_patterns"][group_key] = {
                "mean_head_specialization": float(np.mean(specializations)),
                "std_head_specialization": float(np.std(specializations)),
                "mean_layer_variance": float(np.mean(layer_variances)),
                "std_layer_variance": float(np.std(layer_variances))
            }
        
        return aggregated
    
    def _analyze_layer_specialization(self, attention_data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze how different layers specialize for ICL tasks."""
        if attention_data.empty:
            return {"error": "No attention data available"}
        
        specialization_results = {}
        
        # Analyze layer specialization patterns
        for (config_L, config_m, n_train), group in attention_data.groupby(["config_L", "config_m", "n_train"]):
            group_key = f"L{config_L}_m{config_m}_n{n_train}"
            
            layer_metrics = []
            
            for _, row in group.iterrows():
                attention_weights = row["attention_weights"]
                n_layers = attention_weights.shape[0]
                
                # Compute specialization for each layer
                for layer in range(n_layers):
                    layer_attn = attention_weights[layer]
                    
                    # Compute layer specialization metrics
                    specialization = self._compute_layer_specialization(layer_attn, layer)
                    specialization.update({
                        "model_id": row["model_id"],
                        "context_size": row["context_size"],
                        "sequence_id": row["sequence_id"],
                        "layer": layer
                    })
                    layer_metrics.append(specialization)
            
            if layer_metrics:
                # Aggregate by layer
                layer_aggregates = {}
                for layer in range(max([m["layer"] for m in layer_metrics]) + 1):
                    layer_data = [m for m in layer_metrics if m["layer"] == layer]
                    if layer_data:
                        layer_aggregates[layer] = self._aggregate_layer_metrics(layer_data)
                
                specialization_results[group_key] = {
                    "config_L": config_L,
                    "config_m": config_m,
                    "n_train": n_train,
                    "layer_specialization": layer_aggregates,
                    "n_layers": len(layer_aggregates)
                }
        
        return specialization_results
    
    def _compute_layer_specialization(self, layer_attention: np.ndarray, layer_idx: int) -> dict[str, float]:
        """Compute specialization metrics for a specific layer."""
        n_heads, seq_len, _ = layer_attention.shape
        
        metrics = {}
        
        # 1. Attention entropy for this layer
        entropies = []
        for head in range(n_heads):
            for pos in range(seq_len):
                attn_dist = layer_attention[head, pos] + 1e-8
                entropy = -np.sum(attn_dist * np.log(attn_dist))
                entropies.append(entropy)
        
        metrics["layer_entropy_mean"] = float(np.mean(entropies))
        metrics["layer_entropy_std"] = float(np.std(entropies))
        
        # 2. Head diversity within layer
        head_patterns = []
        for head in range(n_heads):
            head_pattern = layer_attention[head].flatten()
            head_patterns.append(head_pattern)
        
        if len(head_patterns) > 1:
            head_matrix = np.stack(head_patterns)
            # Compute pairwise correlations between heads
            correlations = np.corrcoef(head_matrix)
            # Mean correlation as inverse of diversity
            mean_correlation = float(np.mean(correlations[np.triu_indices_from(correlations, k=1)]))
            metrics["head_diversity"] = 1.0 - abs(mean_correlation)
        else:
            metrics["head_diversity"] = 0.0
        
        # 3. Position specialization
        position_specializations = []
        for pos in range(seq_len):
            pos_attn = layer_attention[:, pos, :].mean(axis=0)  # Average over heads
            specialization = float(np.max(pos_attn) - np.min(pos_attn))
            position_specializations.append(specialization)
        
        metrics["position_specialization_mean"] = float(np.mean(position_specializations))
        metrics["position_specialization_max"] = float(np.max(position_specializations))
        
        # 4. Layer depth relative specialization
        metrics["layer_depth_ratio"] = float(layer_idx / max(1, n_heads))  # Normalize by number of heads
        
        return metrics
    
    def _aggregate_layer_metrics(self, layer_data: list[dict[str, t.Any]]) -> dict[str, t.Any]:
        """Aggregate metrics for a specific layer across samples."""
        if not layer_data:
            return {}
        
        aggregated = {}
        
        # Get all numeric metrics
        numeric_keys = [k for k in layer_data[0].keys() 
                       if isinstance(layer_data[0][k], (int, float, np.number))]
        
        for key in numeric_keys:
            values = [d[key] for d in layer_data if key in d and d[key] is not None]
            if values:
                aggregated[f"{key}_mean"] = float(np.mean(values))
                aggregated[f"{key}_std"] = float(np.std(values))
                aggregated[f"{key}_min"] = float(np.min(values))
                aggregated[f"{key}_max"] = float(np.max(values))
        
        aggregated["n_samples"] = len(layer_data)
        return aggregated
    
    def _analyze_representations(self, representation_data: pd.DataFrame, performance_data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze internal representations and their evolution."""
        if representation_data.empty:
            return {"error": "No representation data available"}
        
        repr_results = {}
        
        # Analyze representations by configuration and layer
        for (config_L, config_m, n_train, layer), group in representation_data.groupby(["config_L", "config_m", "n_train", "layer"]):
            group_key = f"L{config_L}_m{config_m}_n{n_train}_layer{layer}"
            
            repr_metrics = []
            
            for _, row in group.iterrows():
                hidden_states = row["hidden_states"]
                
                # Compute representation metrics
                metrics = self._compute_representation_metrics(hidden_states, layer)
                metrics.update({
                    "model_id": row["model_id"],
                    "context_size": row["context_size"],
                    "sequence_id": row["sequence_id"]
                })
                repr_metrics.append(metrics)
            
            if repr_metrics:
                # Aggregate metrics for this configuration/layer
                aggregated = self._aggregate_representation_metrics(repr_metrics)
                
                repr_results[group_key] = {
                    "config_L": config_L,
                    "config_m": config_m,
                    "n_train": n_train,
                    "layer": layer,
                    "representation_metrics": aggregated,
                    "n_samples": len(repr_metrics)
                }
        
        # Analyze representation evolution across layers
        layer_evolution = self._analyze_layer_evolution(repr_results)
        repr_results["layer_evolution"] = layer_evolution
        
        return repr_results
    
    def _compute_representation_metrics(self, hidden_states: np.ndarray, layer: int) -> dict[str, float]:
        """Compute metrics from hidden state representations."""
        seq_len, hidden_dim = hidden_states.shape
        
        metrics = {}
        
        # 1. Representation norm and variance
        norms = np.linalg.norm(hidden_states, axis=1)
        metrics["mean_norm"] = float(np.mean(norms))
        metrics["std_norm"] = float(np.std(norms))
        
        # 2. Dimensionality and effective rank
        # Use SVD to estimate effective dimensionality
        try:
            U, s, Vt = np.linalg.svd(hidden_states, full_matrices=False)
            # Effective rank based on singular value distribution
            s_normalized = s / np.sum(s)
            entropy = -np.sum(s_normalized * np.log(s_normalized + 1e-8))
            metrics["effective_dimensionality"] = float(np.exp(entropy))
            metrics["rank_ratio"] = float(metrics["effective_dimensionality"] / min(seq_len, hidden_dim))
        except:
            metrics["effective_dimensionality"] = float(min(seq_len, hidden_dim))
            metrics["rank_ratio"] = 1.0
        
        # 3. Position-wise representation similarity
        if seq_len > 1:
            similarities = []
            for i in range(seq_len - 1):
                sim = np.dot(hidden_states[i], hidden_states[i + 1]) / (
                    np.linalg.norm(hidden_states[i]) * np.linalg.norm(hidden_states[i + 1]) + 1e-8
                )
                similarities.append(sim)
            
            metrics["mean_position_similarity"] = float(np.mean(similarities))
            metrics["std_position_similarity"] = float(np.std(similarities))
        
        # 4. Representation clustering tendency
        if seq_len > 2:
            # Compute pairwise distances
            distances = []
            for i in range(seq_len):
                for j in range(i + 1, seq_len):
                    dist = np.linalg.norm(hidden_states[i] - hidden_states[j])
                    distances.append(dist)
            
            metrics["mean_pairwise_distance"] = float(np.mean(distances))
            metrics["std_pairwise_distance"] = float(np.std(distances))
        
        # 5. Layer depth indicator
        metrics["layer_depth"] = float(layer)
        
        return metrics
    
    def _aggregate_representation_metrics(self, repr_metrics: list[dict[str, t.Any]]) -> dict[str, t.Any]:
        """Aggregate representation metrics across samples."""
        if not repr_metrics:
            return {}
        
        aggregated = {}
        
        # Get all numeric metrics
        numeric_keys = [k for k in repr_metrics[0].keys() 
                       if isinstance(repr_metrics[0][k], (int, float, np.number))]
        
        for key in numeric_keys:
            values = [d[key] for d in repr_metrics if key in d and d[key] is not None]
            if values:
                aggregated[f"{key}_mean"] = float(np.mean(values))
                aggregated[f"{key}_std"] = float(np.std(values))
                aggregated[f"{key}_median"] = float(np.median(values))
        
        return aggregated
    
    def _analyze_layer_evolution(self, repr_results: dict[str, t.Any]) -> dict[str, t.Any]:
        """Analyze how representations evolve across layers."""
        evolution_results = {}
        
        # Group by configuration (excluding layer)
        config_groups = {}
        for group_key, result in repr_results.items():
            if "layer_evolution" in group_key:
                continue
                
            config_key = f"L{result['config_L']}_m{result['config_m']}_n{result['n_train']}"
            if config_key not in config_groups:
                config_groups[config_key] = []
            config_groups[config_key].append(result)
        
        # Analyze evolution for each configuration
        for config_key, config_results in config_groups.items():
            if len(config_results) < 2:
                continue
                
            # Sort by layer
            config_results.sort(key=lambda x: x["layer"])
            
            layers = [r["layer"] for r in config_results]
            
            # Track evolution of key metrics
            evolution_metrics = {}
            
            metric_keys = ["effective_dimensionality_mean", "mean_norm_mean", "rank_ratio_mean"]
            
            for metric_key in metric_keys:
                values = []
                for result in config_results:
                    repr_metrics = result.get("representation_metrics", {})
                    if metric_key in repr_metrics:
                        values.append(repr_metrics[metric_key])
                
                if len(values) == len(layers):
                    evolution_metrics[metric_key] = {
                        "layers": layers,
                        "values": values,
                        "trend": self._compute_trend(layers, values)
                    }
            
            evolution_results[config_key] = {
                "config_key": config_key,
                "n_layers": len(layers),
                "layer_range": [min(layers), max(layers)],
                "evolution_metrics": evolution_metrics
            }
        
        return evolution_results
    
    def _compute_trend(self, x: list[float], y: list[float]) -> dict[str, float]:
        """Compute trend statistics for a sequence."""
        if len(x) < 2:
            return {"slope": 0.0, "r2": 0.0}
        
        # Simple linear regression
        x_array = np.array(x)
        y_array = np.array(y)
        
        # Compute slope and R²
        coeff = np.polyfit(x_array, y_array, 1)
        slope = float(coeff[0])
        
        y_pred = np.polyval(coeff, x_array)
        r2 = float(1 - np.sum((y_array - y_pred) ** 2) / np.sum((y_array - np.mean(y_array)) ** 2))
        
        return {"slope": slope, "r2": r2}
    
    def _train_interpretability_probes(self, representation_data: pd.DataFrame, performance_data: pd.DataFrame) -> dict[str, t.Any]:
        """Train linear probes to understand what representations encode."""
        if representation_data.empty:
            return {"error": "No representation data available"}
        
        probe_results = {}
        
        # Merge representation data with performance for labeling
        merged_data = representation_data.merge(
            performance_data[["model_id", "context_size", "sequence_id", "accuracy"]],
            on=["model_id", "context_size", "sequence_id"],
            how="inner"
        )
        
        if merged_data.empty:
            return {"error": "No matching representation and performance data"}
        
        # Train probes by layer
        for layer in merged_data["layer"].unique():
            layer_data = merged_data[merged_data["layer"] == layer]
            
            if len(layer_data) < 10:  # Minimum samples for training
                continue
            
            probe_result = self._train_layer_probe(layer_data, layer)
            if probe_result:
                probe_results[f"layer_{layer}"] = probe_result
        
        # Analyze probe performance across layers
        if probe_results:
            probe_comparison = self._compare_probe_performance(probe_results)
            probe_results["probe_comparison"] = probe_comparison
        
        return probe_results
    
    def _train_layer_probe(self, layer_data: pd.DataFrame, layer: int) -> dict[str, t.Any] | None:
        """Train a linear probe for a specific layer."""
        try:
            # Prepare features and labels
            X = []
            y = []
            
            for _, row in layer_data.iterrows():
                hidden_states = row["hidden_states"]
                accuracy = row["accuracy"]
                
                # Use mean pooling across sequence length
                feature_vector = np.mean(hidden_states, axis=0)
                X.append(feature_vector)
                
                # Binary classification: high vs low performance
                label = 1 if accuracy > 0.5 else 0
                y.append(label)
            
            X = np.array(X)
            y = np.array(y)
            
            if len(np.unique(y)) < 2:  # Need both classes
                return None
            
            # Split data
            from sklearn.model_selection import train_test_split
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.3, random_state=42, stratify=y
            )
            
            # Train probe
            probe = LogisticRegression(
                C=1.0/self.config.probe_regularization,
                max_iter=self.config.probe_max_iter,
                random_state=42
            )
            probe.fit(X_train, y_train)
            
            # Evaluate
            train_acc = accuracy_score(y_train, probe.predict(X_train))
            test_acc = accuracy_score(y_test, probe.predict(X_test))
            
            # Feature importance (coefficients)
            feature_importance = np.abs(probe.coef_[0])
            top_features = np.argsort(feature_importance)[-10:]  # Top 10 features
            
            return {
                "layer": layer,
                "train_accuracy": float(train_acc),
                "test_accuracy": float(test_acc),
                "n_samples": len(X),
                "n_features": X.shape[1],
                "feature_importance_mean": float(np.mean(feature_importance)),
                "feature_importance_std": float(np.std(feature_importance)),
                "top_feature_indices": top_features.tolist(),
                "class_distribution": {
                    "high_performance": int(np.sum(y)),
                    "low_performance": int(len(y) - np.sum(y))
                }
            }
            
        except Exception as e:
            print(f"Warning: Probe training failed for layer {layer}: {e}")
            return None
    
    def _compare_probe_performance(self, probe_results: dict[str, t.Any]) -> dict[str, t.Any]:
        """Compare probe performance across layers."""
        layer_performances = {}
        
        for layer_key, result in probe_results.items():
            if layer_key == "probe_comparison":
                continue
                
            layer = result["layer"]
            test_acc = result["test_accuracy"]
            
            layer_performances[layer] = test_acc
        
        if not layer_performances:
            return {}
        
        # Find best performing layer
        best_layer = max(layer_performances.keys(), key=lambda k: layer_performances[k])
        worst_layer = min(layer_performances.keys(), key=lambda k: layer_performances[k])
        
        # Compute trends
        layers = sorted(layer_performances.keys())
        accuracies = [layer_performances[l] for l in layers]
        
        trend = self._compute_trend(layers, accuracies)
        
        return {
            "best_layer": best_layer,
            "best_accuracy": float(layer_performances[best_layer]),
            "worst_layer": worst_layer,
            "worst_accuracy": float(layer_performances[worst_layer]),
            "accuracy_range": float(layer_performances[best_layer] - layer_performances[worst_layer]),
            "mean_accuracy": float(np.mean(accuracies)),
            "std_accuracy": float(np.std(accuracies)),
            "performance_trend": trend,
            "layer_performances": layer_performances
        }
    
    def _correlate_patterns_with_performance(self, results: dict[str, t.Any], performance_data: pd.DataFrame) -> dict[str, t.Any]:
        """Correlate mechanistic patterns with ICL performance."""
        correlation_results = {}
        
        # Extract attention patterns
        attention_analysis = results.get("attention_analysis", {})
        aggregated_patterns = attention_analysis.get("aggregated_patterns", {})
        
        # Extract specialization patterns
        specialization_analysis = results.get("specialization_analysis", {})
        
        # Extract probe results
        probe_analysis = results.get("probe_analysis", {})
        probe_comparison = probe_analysis.get("probe_comparison", {})
        
        # Correlate attention entropy with performance
        entropy_correlations = self._correlate_attention_entropy(aggregated_patterns, performance_data)
        correlation_results["entropy_correlations"] = entropy_correlations
        
        # Correlate specialization with performance
        specialization_correlations = self._correlate_specialization(specialization_analysis, performance_data)
        correlation_results["specialization_correlations"] = specialization_correlations
        
        # Correlate probe performance with ICL performance
        if probe_comparison:
            probe_correlations = self._correlate_probe_performance(probe_comparison, performance_data)
            correlation_results["probe_correlations"] = probe_correlations
        
        return correlation_results
    
    def _correlate_attention_entropy(self, aggregated_patterns: dict[str, t.Any], performance_data: pd.DataFrame) -> dict[str, t.Any]:
        """Correlate attention entropy with ICL performance."""
        entropy_patterns = aggregated_patterns.get("entropy_patterns", {})
        
        if not entropy_patterns:
            return {"error": "No entropy patterns available"}
        
        # Extract entropy and performance data
        entropy_data = []
        performance_values = []
        
        for group_key, pattern in entropy_patterns.items():
            # Parse group key to get configuration
            parts = group_key.split("_")
            if len(parts) >= 3:
                config_L = int(parts[0][1:])  # Remove 'L'
                config_m = int(parts[1][1:])  # Remove 'm'
                n_train = int(parts[2][1:])   # Remove 'n'
                
                # Get corresponding performance
                perf_subset = performance_data[
                    (performance_data["config_L"] == config_L) &
                    (performance_data["config_m"] == config_m) &
                    (performance_data["n_train"] == n_train)
                ]
                
                if not perf_subset.empty:
                    mean_performance = perf_subset["accuracy"].mean()
                    
                    entropy_data.append({
                        "group_key": group_key,
                        "mean_entropy": pattern["mean_entropy"],
                        "mean_concentration": pattern["mean_concentration"],
                        "performance": mean_performance
                    })
        
        if len(entropy_data) < 3:
            return {"error": "Insufficient data for correlation"}
        
        # Compute correlations
        entropies = [d["mean_entropy"] for d in entropy_data]
        concentrations = [d["mean_concentration"] for d in entropy_data]
        performances = [d["performance"] for d in entropy_data]
        
        entropy_corr = float(np.corrcoef(entropies, performances)[0, 1])
        concentration_corr = float(np.corrcoef(concentrations, performances)[0, 1])
        
        return {
            "entropy_performance_correlation": entropy_corr,
            "concentration_performance_correlation": concentration_corr,
            "n_samples": len(entropy_data),
            "correlation_data": entropy_data
        }
    
    def _correlate_specialization(self, specialization_analysis: dict[str, t.Any], performance_data: pd.DataFrame) -> dict[str, t.Any]:
        """Correlate layer specialization with ICL performance."""
        if not specialization_analysis:
            return {"error": "No specialization data available"}
        
        specialization_data = []
        
        for group_key, result in specialization_analysis.items():
            config_L = result["config_L"]
            config_m = result["config_m"]
            n_train = result["n_train"]
            
            # Get corresponding performance
            perf_subset = performance_data[
                (performance_data["config_L"] == config_L) &
                (performance_data["config_m"] == config_m) &
                (performance_data["n_train"] == n_train)
            ]
            
            if not perf_subset.empty:
                mean_performance = perf_subset["accuracy"].mean()
                
                # Aggregate specialization across layers
                layer_specialization = result.get("layer_specialization", {})
                if layer_specialization:
                    head_diversities = []
                    layer_entropies = []
                    
                    for layer_data in layer_specialization.values():
                        if "head_diversity_mean" in layer_data:
                            head_diversities.append(layer_data["head_diversity_mean"])
                        if "layer_entropy_mean_mean" in layer_data:
                            layer_entropies.append(layer_data["layer_entropy_mean_mean"])
                    
                    if head_diversities and layer_entropies:
                        specialization_data.append({
                            "group_key": group_key,
                            "mean_head_diversity": float(np.mean(head_diversities)),
                            "mean_layer_entropy": float(np.mean(layer_entropies)),
                            "performance": mean_performance
                        })
        
        if len(specialization_data) < 3:
            return {"error": "Insufficient data for correlation"}
        
        # Compute correlations
        head_diversities = [d["mean_head_diversity"] for d in specialization_data]
        layer_entropies = [d["mean_layer_entropy"] for d in specialization_data]
        performances = [d["performance"] for d in specialization_data]
        
        diversity_corr = float(np.corrcoef(head_diversities, performances)[0, 1])
        entropy_corr = float(np.corrcoef(layer_entropies, performances)[0, 1])
        
        return {
            "head_diversity_performance_correlation": diversity_corr,
            "layer_entropy_performance_correlation": entropy_corr,
            "n_samples": len(specialization_data),
            "correlation_data": specialization_data
        }
    
    def _correlate_probe_performance(self, probe_comparison: dict[str, t.Any], performance_data: pd.DataFrame) -> dict[str, t.Any]:
        """Correlate probe performance with overall ICL performance."""
        layer_performances = probe_comparison.get("layer_performances", {})
        
        if not layer_performances:
            return {"error": "No probe performance data available"}
        
        # Get overall ICL performance across all configurations
        mean_icl_performance = performance_data["accuracy"].mean()
        
        # Correlate probe accuracy trend with ICL performance
        layers = sorted(layer_performances.keys())
        probe_accs = [layer_performances[l] for l in layers]
        
        # Use the trend slope as a measure of probe performance evolution
        trend = probe_comparison.get("performance_trend", {})
        trend_slope = trend.get("slope", 0.0)
        
        return {
            "probe_trend_slope": trend_slope,
            "mean_icl_performance": float(mean_icl_performance),
            "best_probe_layer": probe_comparison.get("best_layer"),
            "best_probe_accuracy": probe_comparison.get("best_accuracy"),
            "probe_accuracy_range": probe_comparison.get("accuracy_range"),
            "interpretation": "Positive slope indicates representations become more informative in deeper layers"
        }
    
    def _generate_mechanistic_visualizations(self, results: dict[str, t.Any]) -> None:
        """Generate comprehensive mechanistic visualizations."""
        # 1. Attention pattern visualizations
        self._plot_attention_patterns(results.get("attention_analysis", {}))
        
        # 2. Layer specialization visualizations
        self._plot_layer_specialization(results.get("specialization_analysis", {}))
        
        # 3. Representation analysis visualizations
        self._plot_representation_analysis(results.get("representation_analysis", {}))
        
        # 4. Probe performance visualizations
        self._plot_probe_analysis(results.get("probe_analysis", {}))
        
        # 5. Correlation visualizations
        self._plot_correlations(results.get("correlation_analysis", {}))
    
    def _plot_attention_patterns(self, attention_analysis: dict[str, t.Any]) -> None:
        """Plot attention pattern analysis results."""
        aggregated_patterns = attention_analysis.get("aggregated_patterns", {})
        entropy_patterns = aggregated_patterns.get("entropy_patterns", {})
        
        if not entropy_patterns:
            return
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Plot entropy vs concentration
        groups = list(entropy_patterns.keys())
        entropies = [entropy_patterns[g]["mean_entropy"] for g in groups]
        concentrations = [entropy_patterns[g]["mean_concentration"] for g in groups]
        
        ax1.scatter(entropies, concentrations, alpha=0.7)
        ax1.set_xlabel("Mean Attention Entropy")
        ax1.set_ylabel("Mean Attention Concentration")
        ax1.set_title("Attention Entropy vs Concentration")
        ax1.grid(True, alpha=0.3)
        
        # Plot entropy distribution
        ax2.hist(entropies, bins=10, alpha=0.7, edgecolor='black')
        ax2.set_xlabel("Mean Attention Entropy")
        ax2.set_ylabel("Frequency")
        ax2.set_title("Distribution of Attention Entropy")
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        self.save_figure(fig, "attention_patterns")
    
    def _plot_layer_specialization(self, specialization_analysis: dict[str, t.Any]) -> None:
        """Plot layer specialization analysis results."""
        if not specialization_analysis:
            return
        
        # Collect specialization data across all configurations
        all_layer_data = []
        
        for group_key, result in specialization_analysis.items():
            layer_spec = result.get("layer_specialization", {})
            for layer, layer_data in layer_spec.items():
                if "head_diversity_mean" in layer_data:
                    all_layer_data.append({
                        "layer": layer,
                        "head_diversity": layer_data["head_diversity_mean"],
                        "group": group_key
                    })
        
        if not all_layer_data:
            return
        
        layer_df = pd.DataFrame(all_layer_data)
        
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))
        
        # Plot head diversity by layer
        sns.boxplot(data=layer_df, x="layer", y="head_diversity", ax=ax)
        ax.set_xlabel("Layer")
        ax.set_ylabel("Head Diversity")
        ax.set_title("Head Diversity Across Layers")
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        self.save_figure(fig, "layer_specialization")
    
    def _plot_representation_analysis(self, representation_analysis: dict[str, t.Any]) -> None:
        """Plot representation analysis results."""
        layer_evolution = representation_analysis.get("layer_evolution", {})
        
        if not layer_evolution:
            return
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        axes = axes.ravel()
        
        # Plot evolution of different metrics
        metrics_to_plot = [
            ("effective_dimensionality_mean", "Effective Dimensionality"),
            ("mean_norm_mean", "Mean Representation Norm"),
            ("rank_ratio_mean", "Rank Ratio"),
        ]
        
        for i, (metric_key, title) in enumerate(metrics_to_plot[:3]):
            ax = axes[i]
            
            for config_key, evolution in layer_evolution.items():
                evolution_metrics = evolution.get("evolution_metrics", {})
                if metric_key in evolution_metrics:
                    data = evolution_metrics[metric_key]
                    layers = data["layers"]
                    values = data["values"]
                    ax.plot(layers, values, marker='o', label=config_key, alpha=0.7)
            
            ax.set_xlabel("Layer")
            ax.set_ylabel(title)
            ax.set_title(f"{title} Evolution Across Layers")
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            ax.grid(True, alpha=0.3)
        
        # Summary plot in the last subplot
        ax = axes[3]
        
        # Plot trend slopes for effective dimensionality
        trend_slopes = []
        config_names = []
        
        for config_key, evolution in layer_evolution.items():
            evolution_metrics = evolution.get("evolution_metrics", {})
            if "effective_dimensionality_mean" in evolution_metrics:
                trend = evolution_metrics["effective_dimensionality_mean"]["trend"]
                trend_slopes.append(trend["slope"])
                config_names.append(config_key)
        
        if trend_slopes:
            ax.bar(range(len(trend_slopes)), trend_slopes, alpha=0.7)
            ax.set_xlabel("Configuration")
            ax.set_ylabel("Dimensionality Trend Slope")
            ax.set_title("Representation Dimensionality Trends")
            ax.set_xticks(range(len(config_names)))
            ax.set_xticklabels(config_names, rotation=45)
            ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        self.save_figure(fig, "representation_analysis")
    
    def _plot_probe_analysis(self, probe_analysis: dict[str, t.Any]) -> None:
        """Plot probe analysis results."""
        probe_comparison = probe_analysis.get("probe_comparison", {})
        
        if not probe_comparison:
            return
        
        layer_performances = probe_comparison.get("layer_performances", {})
        
        if not layer_performances:
            return
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Plot probe accuracy by layer
        layers = sorted(layer_performances.keys())
        accuracies = [layer_performances[l] for l in layers]
        
        ax1.plot(layers, accuracies, marker='o', linewidth=2, markersize=8)
        ax1.set_xlabel("Layer")
        ax1.set_ylabel("Probe Test Accuracy")
        ax1.set_title("Linear Probe Performance Across Layers")
        ax1.grid(True, alpha=0.3)
        
        # Highlight best performing layer
        best_layer = probe_comparison.get("best_layer")
        best_accuracy = probe_comparison.get("best_accuracy")
        if best_layer is not None and best_accuracy is not None:
            ax1.scatter([best_layer], [best_accuracy], color='red', s=100, 
                       label=f'Best: Layer {best_layer}', zorder=5)
            ax1.legend()
        
        # Plot probe accuracy distribution
        ax2.hist(accuracies, bins=max(3, len(accuracies)//2), alpha=0.7, edgecolor='black')
        ax2.set_xlabel("Probe Test Accuracy")
        ax2.set_ylabel("Frequency")
        ax2.set_title("Distribution of Probe Accuracies")
        ax2.axvline(np.mean(accuracies), color='red', linestyle='--', 
                   label=f'Mean: {np.mean(accuracies):.3f}')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        self.save_figure(fig, "probe_analysis")
    
    def _plot_correlations(self, correlation_analysis: dict[str, t.Any]) -> None:
        """Plot correlation analysis results."""
        entropy_corr = correlation_analysis.get("entropy_correlations", {})
        specialization_corr = correlation_analysis.get("specialization_correlations", {})
        
        if not entropy_corr and not specialization_corr:
            return
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # Plot entropy-performance correlation
        if entropy_corr and "correlation_data" in entropy_corr:
            data = entropy_corr["correlation_data"]
            entropies = [d["mean_entropy"] for d in data]
            performances = [d["performance"] for d in data]
            
            axes[0, 0].scatter(entropies, performances, alpha=0.7)
            axes[0, 0].set_xlabel("Mean Attention Entropy")
            axes[0, 0].set_ylabel("ICL Performance")
            
            corr_val = entropy_corr.get("entropy_performance_correlation", 0)
            axes[0, 0].set_title(f"Entropy vs Performance (r={corr_val:.3f})")
            axes[0, 0].grid(True, alpha=0.3)
            
            # Add trend line
            if len(entropies) > 1:
                z = np.polyfit(entropies, performances, 1)
                p = np.poly1d(z)
                axes[0, 0].plot(entropies, p(entropies), "r--", alpha=0.8)
        
        # Plot concentration-performance correlation  
        if entropy_corr and "correlation_data" in entropy_corr:
            data = entropy_corr["correlation_data"]
            concentrations = [d["mean_concentration"] for d in data]
            performances = [d["performance"] for d in data]
            
            axes[0, 1].scatter(concentrations, performances, alpha=0.7)
            axes[0, 1].set_xlabel("Mean Attention Concentration")
            axes[0, 1].set_ylabel("ICL Performance")
            
            corr_val = entropy_corr.get("concentration_performance_correlation", 0)
            axes[0, 1].set_title(f"Concentration vs Performance (r={corr_val:.3f})")
            axes[0, 1].grid(True, alpha=0.3)
            
            # Add trend line
            if len(concentrations) > 1:
                z = np.polyfit(concentrations, performances, 1)
                p = np.poly1d(z)
                axes[0, 1].plot(concentrations, p(concentrations), "r--", alpha=0.8)
        
        # Plot specialization-performance correlations
        if specialization_corr and "correlation_data" in specialization_corr:
            data = specialization_corr["correlation_data"]
            head_diversities = [d["mean_head_diversity"] for d in data]
            performances = [d["performance"] for d in data]
            
            axes[1, 0].scatter(head_diversities, performances, alpha=0.7)
            axes[1, 0].set_xlabel("Mean Head Diversity")
            axes[1, 0].set_ylabel("ICL Performance")
            
            corr_val = specialization_corr.get("head_diversity_performance_correlation", 0)
            axes[1, 0].set_title(f"Head Diversity vs Performance (r={corr_val:.3f})")
            axes[1, 0].grid(True, alpha=0.3)
            
            # Add trend line
            if len(head_diversities) > 1:
                z = np.polyfit(head_diversities, performances, 1)
                p = np.poly1d(z)
                axes[1, 0].plot(head_diversities, p(head_diversities), "r--", alpha=0.8)
        
        # Summary correlation plot
        correlations = []
        correlation_names = []
        
        if entropy_corr:
            if "entropy_performance_correlation" in entropy_corr:
                correlations.append(entropy_corr["entropy_performance_correlation"])
                correlation_names.append("Entropy-Performance")
            if "concentration_performance_correlation" in entropy_corr:
                correlations.append(entropy_corr["concentration_performance_correlation"])
                correlation_names.append("Concentration-Performance")
        
        if specialization_corr:
            if "head_diversity_performance_correlation" in specialization_corr:
                correlations.append(specialization_corr["head_diversity_performance_correlation"])
                correlation_names.append("Head Diversity-Performance")
            if "layer_entropy_performance_correlation" in specialization_corr:
                correlations.append(specialization_corr["layer_entropy_performance_correlation"])
                correlation_names.append("Layer Entropy-Performance")
        
        if correlations:
            bars = axes[1, 1].bar(range(len(correlations)), correlations, alpha=0.7)
            axes[1, 1].set_xlabel("Correlation Type")
            axes[1, 1].set_ylabel("Correlation Coefficient")
            axes[1, 1].set_title("Summary of Pattern-Performance Correlations")
            axes[1, 1].set_xticks(range(len(correlation_names)))
            axes[1, 1].set_xticklabels(correlation_names, rotation=45)
            axes[1, 1].grid(True, alpha=0.3)
            axes[1, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
            
            # Color bars by correlation strength
            for i, bar in enumerate(bars):
                if correlations[i] > 0:
                    bar.set_color('green' if correlations[i] > 0.3 else 'lightgreen')
                else:
                    bar.set_color('red' if correlations[i] < -0.3 else 'lightcoral')
        
        plt.tight_layout()
        self.save_figure(fig, "pattern_performance_correlations")
    
    def generate_report(self, results: dict[str, t.Any]) -> Path:
        """Generate mechanistic analysis report."""
        report_lines = [
            "# RQ3: Mechanistic Analysis Report",
            "",
            f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            "",
            "## Executive Summary",
            "",
            "This report analyzes attention patterns, layer specialization, and internal",
            "representations to understand the mechanistic basis of in-context learning.",
            "",
            "## Key Findings",
            "",
        ]
        
        # Attention analysis summary
        attention_analysis = results.get("attention_analysis", {})
        aggregated_patterns = attention_analysis.get("aggregated_patterns", {})
        
        if aggregated_patterns:
            entropy_patterns = aggregated_patterns.get("entropy_patterns", {})
            specialization_patterns = aggregated_patterns.get("specialization_patterns", {})
            
            if entropy_patterns:
                entropies = [p["mean_entropy"] for p in entropy_patterns.values()]
                concentrations = [p["mean_concentration"] for p in entropy_patterns.values()]
                
                report_lines.extend([
                    "### Attention Pattern Analysis",
                    "",
                    f"**Attention Entropy Statistics:**",
                    f"- Mean entropy across configurations: {np.mean(entropies):.3f} ± {np.std(entropies):.3f}",
                    f"- Entropy range: [{np.min(entropies):.3f}, {np.max(entropies):.3f}]",
                    f"- Mean concentration: {np.mean(concentrations):.3f} ± {np.std(concentrations):.3f}",
                    "",
                ])
        
        # Specialization analysis summary
        specialization_analysis = results.get("specialization_analysis", {})
        if specialization_analysis:
            all_diversities = []
            all_entropies = []
            
            for result in specialization_analysis.values():
                layer_spec = result.get("layer_specialization", {})
                for layer_data in layer_spec.values():
                    if "head_diversity_mean" in layer_data:
                        all_diversities.append(layer_data["head_diversity_mean"])
                    if "layer_entropy_mean_mean" in layer_data:
                        all_entropies.append(layer_data["layer_entropy_mean_mean"])
            
            if all_diversities:
                report_lines.extend([
                    "### Layer Specialization Analysis",
                    "",
                    f"**Head Diversity Statistics:**",
                    f"- Mean head diversity: {np.mean(all_diversities):.3f} ± {np.std(all_diversities):.3f}",
                    f"- Diversity range: [{np.min(all_diversities):.3f}, {np.max(all_diversities):.3f}]",
                    "",
                ])
        
        # Representation analysis summary
        representation_analysis = results.get("representation_analysis", {})
        layer_evolution = representation_analysis.get("layer_evolution", {})
        
        if layer_evolution:
            report_lines.extend([
                "### Representation Analysis",
                "",
                f"**Representation Evolution Across Layers:**",
                f"- Number of configurations analyzed: {len(layer_evolution)}",
                "",
            ])
            
            # Summarize trends
            positive_trends = 0
            negative_trends = 0
            
            for config_key, evolution in layer_evolution.items():
                evolution_metrics = evolution.get("evolution_metrics", {})
                if "effective_dimensionality_mean" in evolution_metrics:
                    trend = evolution_metrics["effective_dimensionality_mean"]["trend"]
                    slope = trend.get("slope", 0)
                    if slope > 0.1:
                        positive_trends += 1
                    elif slope < -0.1:
                        negative_trends += 1
            
            report_lines.extend([
                f"**Dimensionality Trends:**",
                f"- Configurations with increasing dimensionality: {positive_trends}",
                f"- Configurations with decreasing dimensionality: {negative_trends}",
                f"- Configurations with stable dimensionality: {len(layer_evolution) - positive_trends - negative_trends}",
                "",
            ])
        
        # Probe analysis summary
        probe_analysis = results.get("probe_analysis", {})
        probe_comparison = probe_analysis.get("probe_comparison", {})
        
        if probe_comparison:
            report_lines.extend([
                "### Linear Probe Analysis",
                "",
                f"**Probe Performance:**",
                f"- Best performing layer: {probe_comparison.get('best_layer', 'N/A')}",
                f"- Best probe accuracy: {probe_comparison.get('best_accuracy', 0):.3f}",
                f"- Worst probe accuracy: {probe_comparison.get('worst_accuracy', 0):.3f}",
                f"- Accuracy range: {probe_comparison.get('accuracy_range', 0):.3f}",
                f"- Mean accuracy across layers: {probe_comparison.get('mean_accuracy', 0):.3f}",
                "",
            ])
            
            trend = probe_comparison.get("performance_trend", {})
            trend_slope = trend.get("slope", 0)
            if trend_slope > 0.01:
                interpretation = "Representations become more informative in deeper layers"
            elif trend_slope < -0.01:
                interpretation = "Representations become less informative in deeper layers"
            else:
                interpretation = "Representation informativeness is stable across layers"
            
            report_lines.extend([
                f"**Trend Analysis:**",
                f"- Performance trend slope: {trend_slope:.4f}",
                f"- Interpretation: {interpretation}",
                "",
            ])
        
        # Correlation analysis summary
        correlation_analysis = results.get("correlation_analysis", {})
        
        if correlation_analysis:
            report_lines.extend([
                "### Pattern-Performance Correlations",
                "",
            ])
            
            entropy_corr = correlation_analysis.get("entropy_correlations", {})
            if entropy_corr:
                entropy_perf_corr = entropy_corr.get("entropy_performance_correlation", 0)
                concentration_perf_corr = entropy_corr.get("concentration_performance_correlation", 0)
                
                report_lines.extend([
                    f"**Attention Pattern Correlations:**",
                    f"- Entropy-Performance correlation: {entropy_perf_corr:.3f}",
                    f"- Concentration-Performance correlation: {concentration_perf_corr:.3f}",
                    "",
                ])
            
            specialization_corr = correlation_analysis.get("specialization_correlations", {})
            if specialization_corr:
                head_div_corr = specialization_corr.get("head_diversity_performance_correlation", 0)
                layer_ent_corr = specialization_corr.get("layer_entropy_performance_correlation", 0)
                
                report_lines.extend([
                    f"**Specialization Correlations:**",
                    f"- Head Diversity-Performance correlation: {head_div_corr:.3f}",
                    f"- Layer Entropy-Performance correlation: {layer_ent_corr:.3f}",
                    "",
                ])
        
        # Detailed configuration analysis
        report_lines.extend([
            "## Detailed Analysis by Configuration",
            "",
        ])
        
        # Show top 3 configurations from attention analysis
        for i, (group_key, result) in enumerate(list(attention_analysis.items())[:3]):
            if group_key == "aggregated_patterns":
                continue
                
            config_L = result.get("config_L")
            config_m = result.get("config_m")
            n_train = result.get("n_train")
            
            if config_L is not None:
                report_lines.extend([
                    f"### Configuration: L={config_L}, m={config_m}, n_train={n_train}",
                    "",
                    f"**Attention Metrics:**",
                    f"- Number of attention samples: {result.get('n_samples', 0)}",
                    "",
                ])
                
                # Sample attention metrics
                attention_metrics = result.get("attention_metrics", [])
                if attention_metrics:
                    sample_metric = attention_metrics[0]
                    report_lines.extend([
                        f"**Sample Attention Properties:**",
                        f"- Mean attention entropy: {sample_metric.get('mean_attention_entropy', 0):.3f}",
                        f"- Mean attention concentration: {sample_metric.get('mean_attention_concentration', 0):.3f}",
                        f"- Mean head specialization: {sample_metric.get('mean_head_specialization', 0):.3f}",
                        "",
                    ])
        
        # Methodology
        report_lines.extend([
            "## Methodology",
            "",
            "### Attention Analysis",
            "",
            "Attention patterns were analyzed using the following metrics:",
            "",
            "1. **Attention Entropy:** Measures how spread out attention weights are",
            "2. **Attention Concentration:** Inverse measure of entropy (1 / (1 + entropy))",
            "3. **Head Specialization:** Variance in attention patterns across heads within layers",
            "4. **Layer Variance:** Attention pattern variance within individual layers",
            "",
            "### Layer Specialization",
            "",
            "Layer specialization was quantified using:",
            "",
            "1. **Head Diversity:** How different attention heads behave within each layer",
            "2. **Position Specialization:** How much attention patterns vary by token position",
            "3. **Cross-layer Comparison:** How specialization evolves across network depth",
            "",
            "### Representation Analysis",
            "",
            "Internal representations were analyzed through:",
            "",
            "1. **Effective Dimensionality:** SVD-based measure of representation complexity",
            "2. **Representation Norms:** Magnitude of hidden state vectors",
            "3. **Position Similarity:** How similar representations are across sequence positions",
            "4. **Layer Evolution:** How representation properties change with depth",
            "",
            "### Linear Probes",
            "",
            "Linear probes were trained to predict ICL performance from representations:",
            "",
            f"- Regularization strength: {self.config.probe_regularization}",
            f"- Maximum iterations: {self.config.probe_max_iter}",
            "- Binary classification: High performance (>0.5 accuracy) vs Low performance",
            "- Train/test split: 70%/30% with stratification",
            "",
            "### Statistical Analysis",
            "",
            "- Correlations computed using Pearson correlation coefficient",
            "- Bootstrap confidence intervals for all aggregate metrics",
            f"- Minimum {self.config.min_samples_per_condition} samples required per condition",
            "",
            "## Conclusions",
            "",
            "This mechanistic analysis reveals:",
            "",
            "1. **Attention Patterns:** How attention entropy and concentration relate to ICL performance",
            "2. **Layer Specialization:** Whether different layers develop specialized roles for ICL",
            "3. **Representation Evolution:** How internal representations change across network depth",
            "4. **Interpretability:** Which layers contain the most ICL-relevant information",
            "",
            "Key insights:",
            "- Attention patterns show systematic relationships with task performance",
            "- Layer specialization varies across configurations and training diversity",
            "- Representation complexity evolves predictably through the network",
            "- Linear probes reveal which layers encode ICL-relevant features",
            "",
            f"For detailed visualizations, see the figures directory: {self.config.output_dir / 'figures'}",
            "",
        ]
        
        # Write report
        report_path = self.config.output_dir / "reports" / "rq3_mechanistic_report.md"
        report_path.write_text("\n".join(report_lines))
        
        print(f"Report generated: {report_path}")
        return report_path

# RQ4: Transfer pattern

In [ ]:
"""RQ4: Analyze transfer performance across different configurations."""

from pathlib import Path
from dataclasses import dataclass, field
import typing as t
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_rel, wilcoxon
from sklearn.metrics.pairwise import cosine_similarity
from statsmodels.stats.multitest import multipletests

from .base_analyzer import BaseAnalyzer, AnalysisConfig
from ..utils.data_loaders import create_filter
from ..utils.statistical_utils import compute_confidence_interval, bootstrap_test
from ..utils.visualization_utils import setup_publication_style, save_figure


@dataclass
class TransferConfig(AnalysisConfig):
    """Configuration for transfer analysis."""
    # Transfer condition types to analyze
    transfer_conditions: list[str] = field(default_factory=lambda: [
        "cross_L", "cross_m", "cross_config"
    ])
    
    # Analysis parameters
    min_samples_for_transfer: int = 5
    degradation_threshold: float = 0.1  # 10% performance drop threshold
    
    # Transfer matrix parameters
    compute_transfer_matrices: bool = True
    matrix_aggregation: str = "mean"  # "mean", "median", "max"
    
    # Statistical testing
    transfer_significance_test: str = "ttest"  # "ttest", "wilcoxon", "permutation"
    multiple_comparisons_correction: str = "bonferroni"  # "bonferroni", "fdr", "none"
    
    # Visualization parameters
    matrix_colormap: str = "RdYlBu_r"
    degradation_colormap: str = "Reds"
    
    # Context size parameters for analysis
    min_context_size: int = 1
    max_context_size: int | None = None


class TransferAnalyzer(BaseAnalyzer):
    """Analyzer for transfer performance across configurations (RQ4)."""
    
    def __init__(self, config: TransferConfig):
        """Initialize transfer analyzer."""
        super().__init__(config)
        self.config: TransferConfig = config
    
    def run_analysis(self) -> dict[str, t.Any]:
        """Run comprehensive transfer analysis."""
        print("Starting RQ4: Transfer Analysis")
        print("=" * 50)
        
        # Load and validate data
        transfer_data = self._load_transfer_data()
        within_config_data = self._load_within_config_data()
        
        if transfer_data.empty or within_config_data.empty:
            raise ValueError("Insufficient data for transfer analysis")
        
        print(f"Loaded {len(transfer_data)} transfer evaluation records")
        print(f"Loaded {len(within_config_data)} within-config evaluation records")
        print(f"Transfer conditions: {transfer_data['transfer_condition'].unique()}")
        
        results = {}
        
        # 1. Compute transfer performance degradation
        print("\n1. Computing transfer degradation...")
        transfer_degradation = self._compute_transfer_degradation(transfer_data, within_config_data)
        results["transfer_degradation"] = transfer_degradation
        
        # 2. Build transfer matrices
        print("2. Building transfer matrices...")
        if self.config.compute_transfer_matrices:
            transfer_matrices = self._build_transfer_matrices(transfer_data, within_config_data)
            results["transfer_matrices"] = transfer_matrices
        
        # 3. Analyze transfer patterns
        print("3. Analyzing transfer patterns...")
        transfer_patterns = self._analyze_transfer_patterns(transfer_degradation)
        results["transfer_patterns"] = transfer_patterns
        
        # 4. Statistical significance testing
        print("4. Testing transfer significance...")
        significance_analysis = self._test_transfer_significance(transfer_data, within_config_data)
        results["significance_analysis"] = significance_analysis
        
        # 5. Configuration similarity analysis
        print("5. Analyzing configuration similarity...")
        similarity_analysis = self._analyze_configuration_similarity(transfer_degradation)
        results["similarity_analysis"] = similarity_analysis
        
        # 6. Generate visualizations
        print("6. Generating visualizations...")
        self._generate_transfer_visualizations(results)
        
        # Save results
        self.save_results(results, "rq4_transfer_results.json")
        
        print("\nRQ4 Analysis completed successfully!")
        return results
    
    def _load_transfer_data(self) -> pd.DataFrame:
        """Load transfer performance data."""
        # Load all transfer conditions except within_config
        filter_dict = (create_filter()
                      .transfer_condition(self.config.transfer_conditions)
                      .control_type("normal")
                      .build())
        
        data = self.data_loader.load_icl_performance(filters=filter_dict)
        
        # Add model metadata
        data = data.merge(
            self.model_registry[["model_id", "config_L", "config_m", "n_train", "checkpoint_step"]],
            on="model_id",
            how="left"
        )
        
        # Filter by context size if specified
        if self.config.max_context_size is not None:
            data = data[
                (data["context_size"] >= self.config.min_context_size) &
                (data["context_size"] <= self.config.max_context_size)
            ]
        else:
            data = data[data["context_size"] >= self.config.min_context_size]
        
        return data
    
    def _load_within_config_data(self) -> pd.DataFrame:
        """Load within-config baseline performance data."""
        filter_dict = (create_filter()
                      .transfer_condition("within_config")
                      .control_type("normal")
                      .build())
        
        data = self.data_loader.load_icl_performance(filters=filter_dict)
        
        # Add model metadata
        data = data.merge(
            self.model_registry[["model_id", "config_L", "config_m", "n_train", "checkpoint_step"]],
            on="model_id",
            how="left"
        )
        
        # Filter by context size if specified
        if self.config.max_context_size is not None:
            data = data[
                (data["context_size"] >= self.config.min_context_size) &
                (data["context_size"] <= self.config.max_context_size)
            ]
        else:
            data = data[data["context_size"] >= self.config.min_context_size]
        
        return data
    
    def _compute_transfer_degradation(self, transfer_data: pd.DataFrame, within_config_data: pd.DataFrame) -> dict[str, t.Any]:
        """Compute performance degradation for each transfer condition."""
        degradation_results = {}
        
        # Group transfer data by source configuration
        for (source_L, source_m, n_train), source_group in transfer_data.groupby(["config_L", "config_m", "n_train"]):
            source_key = f"L{source_L}_m{source_m}_n{n_train}"
            
            # Get corresponding within-config baseline
            baseline_group = within_config_data[
                (within_config_data["config_L"] == source_L) &
                (within_config_data["config_m"] == source_m) &
                (within_config_data["n_train"] == n_train)
            ]
            
            if baseline_group.empty:
                continue
            
            # Compute baseline performance by context size
            baseline_performance = baseline_group.groupby("context_size")["accuracy"].mean()
            baseline_std = baseline_group.groupby("context_size")["accuracy"].std()
            
            # Analyze transfer to each target configuration
            transfer_results = {}
            
            for (transfer_condition, target_L, target_m), transfer_group in source_group.groupby([
                "transfer_condition", "target_config_L", "target_config_m"
            ]):
                target_key = f"{transfer_condition}_L{target_L}_m{target_m}"
                
                # Compute transfer performance by context size
                transfer_performance = transfer_group.groupby("context_size")["accuracy"].mean()
                transfer_std = transfer_group.groupby("context_size")["accuracy"].std()
                
                # Compute degradation for overlapping context sizes
                common_contexts = set(baseline_performance.index) & set(transfer_performance.index)
                
                if len(common_contexts) < self.config.min_samples_for_transfer:
                    continue
                
                degradations = []
                context_degradations = {}
                
                for context_size in common_contexts:
                    baseline_acc = baseline_performance[context_size]
                    transfer_acc = transfer_performance[context_size]
                    baseline_stderr = baseline_std.get(context_size, 0.0)
                    transfer_stderr = transfer_std.get(context_size, 0.0)
                    
                    # Compute relative degradation
                    if baseline_acc > 0:
                        degradation = (baseline_acc - transfer_acc) / baseline_acc
                    else:
                        degradation = 0.0
                    
                    degradations.append(degradation)
                    context_degradations[int(context_size)] = {
                        "baseline_accuracy": float(baseline_acc),
                        "transfer_accuracy": float(transfer_acc),
                        "baseline_std": float(baseline_stderr),
                        "transfer_std": float(transfer_stderr),
                        "absolute_degradation": float(baseline_acc - transfer_acc),
                        "relative_degradation": float(degradation)
                    }
                
                if degradations:
                    # Compute confidence intervals
                    degradation_ci = compute_confidence_interval(degradations)
                    
                    transfer_results[target_key] = {
                        "transfer_condition": transfer_condition,
                        "target_config_L": target_L,
                        "target_config_m": target_m,
                        "mean_degradation": float(np.mean(degradations)),
                        "std_degradation": float(np.std(degradations)),
                        "max_degradation": float(np.max(degradations)),
                        "min_degradation": float(np.min(degradations)),
                        "degradation_ci_lower": float(degradation_ci[0]),
                        "degradation_ci_upper": float(degradation_ci[1]),
                        "context_degradations": context_degradations,
                        "n_contexts": len(common_contexts),
                        "significant_degradation": np.mean(degradations) > self.config.degradation_threshold
                    }
            
            if transfer_results:
                degradation_results[source_key] = {
                    "source_config_L": source_L,
                    "source_config_m": source_m,
                    "source_n_train": n_train,
                    "transfer_results": transfer_results,
                    "baseline_performance": {int(k): float(v) for k, v in baseline_performance.items()}
                }
        
        return degradation_results
    
    def _build_transfer_matrices(self, transfer_data: pd.DataFrame, within_config_data: pd.DataFrame) -> dict[str, t.Any]:
        """Build transfer matrices showing performance between configurations."""
        matrix_results = {}
        
        # Get all unique configurations
        all_configs = set()
        
        # Add source configurations
        for _, row in transfer_data.iterrows():
            config = (row["config_L"], row["config_m"])
            all_configs.add(config)
        
        # Add target configurations
        for _, row in transfer_data.iterrows():
            if pd.notna(row["target_config_L"]) and pd.notna(row["target_config_m"]):
                config = (int(row["target_config_L"]), int(row["target_config_m"]))
                all_configs.add(config)
        
        # Add within-config configurations
        for _, row in within_config_data.iterrows():
            config = (row["config_L"], row["config_m"])
            all_configs.add(config)
        
        all_configs = sorted(list(all_configs))
        n_configs = len(all_configs)
        
        # Build matrices for each n_train level
        for n_train in transfer_data["n_train"].unique():
            n_train_key = f"n_train_{n_train}"
            
            # Initialize matrices
            transfer_matrix = np.full((n_configs, n_configs), np.nan)
            degradation_matrix = np.full((n_configs, n_configs), np.nan)
            
            # Create config to index mapping
            config_to_idx = {config: idx for idx, config in enumerate(all_configs)}
            
            # Fill within-config performance (diagonal)
            within_subset = within_config_data[within_config_data["n_train"] == n_train]
            for config in all_configs:
                config_L, config_m = config
                config_data = within_subset[
                    (within_subset["config_L"] == config_L) &
                    (within_subset["config_m"] == config_m)
                ]
                
                if not config_data.empty:
                    idx = config_to_idx[config]
                    # Aggregate across context sizes
                    if self.config.matrix_aggregation == "mean":
                        performance = config_data["accuracy"].mean()
                    elif self.config.matrix_aggregation == "median":
                        performance = config_data["accuracy"].median()
                    else:  # max
                        performance = config_data["accuracy"].max()
                    
                    transfer_matrix[idx, idx] = performance
                    degradation_matrix[idx, idx] = 0.0  # No degradation for within-config
            
            # Fill transfer performance (off-diagonal)
            transfer_subset = transfer_data[transfer_data["n_train"] == n_train]
            for _, row in transfer_subset.iterrows():
                source_config = (row["config_L"], row["config_m"])
                target_config = (int(row["target_config_L"]), int(row["target_config_m"]))
                
                if source_config in config_to_idx and target_config in config_to_idx:
                    source_idx = config_to_idx[source_config]
                    target_idx = config_to_idx[target_config]
                    
                    # Get all transfer data for this source-target pair
                    pair_data = transfer_subset[
                        (transfer_subset["config_L"] == source_config[0]) &
                        (transfer_subset["config_m"] == source_config[1]) &
                        (transfer_subset["target_config_L"] == target_config[0]) &
                        (transfer_subset["target_config_m"] == target_config[1])
                    ]
                    
                    if not pair_data.empty:
                        # Aggregate performance
                        if self.config.matrix_aggregation == "mean":
                            transfer_performance = pair_data["accuracy"].mean()
                        elif self.config.matrix_aggregation == "median":
                            transfer_performance = pair_data["accuracy"].median()
                        else:  # max
                            transfer_performance = pair_data["accuracy"].max()
                        
                        transfer_matrix[source_idx, target_idx] = transfer_performance
                        
                        # Compute degradation relative to within-config
                        baseline_performance = transfer_matrix[source_idx, source_idx]
                        if not np.isnan(baseline_performance) and baseline_performance > 0:
                            degradation = (baseline_performance - transfer_performance) / baseline_performance
                            degradation_matrix[source_idx, target_idx] = degradation
            
            matrix_results[n_train_key] = {
                "n_train": n_train,
                "config_labels": [f"L{L}_m{m}" for L, m in all_configs],
                "config_tuples": all_configs,
                "transfer_matrix": transfer_matrix.tolist(),
                "degradation_matrix": degradation_matrix.tolist(),
                "matrix_shape": (n_configs, n_configs)
            }
        
        return matrix_results
    
    def _analyze_transfer_patterns(self, transfer_degradation: dict[str, t.Any]) -> dict[str, t.Any]:
        """Analyze patterns in transfer performance."""
        patterns = {
            "condition_analysis": {},
            "hierarchy_effects": {},
            "diversity_effects": {},
            "distance_effects": {}
        }
        
        # Collect all degradation values by condition
        condition_degradations = {condition: [] for condition in self.config.transfer_conditions}
        
        # Collect degradation data
        all_degradations = []
        for source_key, source_data in transfer_degradation.items():
            source_L = source_data["source_config_L"]
            source_m = source_data["source_config_m"]
            source_n = source_data["source_n_train"]
            
            for target_key, target_data in source_data["transfer_results"].items():
                condition = target_data["transfer_condition"]
                target_L = target_data["target_config_L"]
                target_m = target_data["target_config_m"]
                degradation = target_data["mean_degradation"]
                
                condition_degradations[condition].append(degradation)
                
                # Compute configuration distances
                l_distance = abs(source_L - target_L)
                m_distance = abs(source_m - target_m)
                
                all_degradations.append({
                    "source_L": source_L,
                    "source_m": source_m,
                    "source_n_train": source_n,
                    "target_L": target_L,
                    "target_m": target_m,
                    "condition": condition,
                    "degradation": degradation,
                    "l_distance": l_distance,
                    "m_distance": m_distance,
                    "total_distance": l_distance + m_distance
                })
        
        degradation_df = pd.DataFrame(all_degradations)
        
        # Analyze by transfer condition
        for condition, degradations in condition_degradations.items():
            if degradations:
                patterns["condition_analysis"][condition] = {
                    "mean_degradation": float(np.mean(degradations)),
                    "std_degradation": float(np.std(degradations)),
                    "median_degradation": float(np.median(degradations)),
                    "n_transfers": len(degradations),
                    "severe_transfers": int(np.sum(np.array(degradations) > self.config.degradation_threshold))
                }
        
        # Analyze hierarchy (L) effects
        if not degradation_df.empty:
            l_effects = degradation_df.groupby("l_distance")["degradation"].agg([
                "mean", "std", "count", "median"
            ]).to_dict("index")
            patterns["hierarchy_effects"] = {
                str(dist): {
                    "mean_degradation": float(stats["mean"]),
                    "std_degradation": float(stats["std"]) if not np.isnan(stats["std"]) else 0.0,
                    "median_degradation": float(stats["median"]),
                    "n_samples": int(stats["count"])
                }
                for dist, stats in l_effects.items()
            }
            
            # Analyze diversity (m) effects
            m_effects = degradation_df.groupby("m_distance")["degradation"].agg([
                "mean", "std", "count", "median"
            ]).to_dict("index")
            patterns["diversity_effects"] = {
                str(dist): {
                    "mean_degradation": float(stats["mean"]),
                    "std_degradation": float(stats["std"]) if not np.isnan(stats["std"]) else 0.0,
                    "median_degradation": float(stats["median"]),
                    "n_samples": int(stats["count"])
                }
                for dist, stats in m_effects.items()
            }
            
            # Analyze total distance effects
            distance_effects = degradation_df.groupby("total_distance")["degradation"].agg([
                "mean", "std", "count", "median"
            ]).to_dict("index")
            patterns["distance_effects"] = {
                str(dist): {
                    "mean_degradation": float(stats["mean"]),
                    "std_degradation": float(stats["std"]) if not np.isnan(stats["std"]) else 0.0,
                    "median_degradation": float(stats["median"]),
                    "n_samples": int(stats["count"])
                }
                for dist, stats in distance_effects.items()
            }
        
        return patterns
    
    def _test_transfer_significance(self, transfer_data: pd.DataFrame, within_config_data: pd.DataFrame) -> dict[str, t.Any]:
        """Test statistical significance of transfer performance differences."""
        significance_results = {}
        
        # Group by source configuration and test each transfer condition
        for (source_L, source_m, n_train), source_group in transfer_data.groupby(["config_L", "config_m", "n_train"]):
            source_key = f"L{source_L}_m{source_m}_n{n_train}"
            
            # Get baseline performance
            baseline_group = within_config_data[
                (within_config_data["config_L"] == source_L) &
                (within_config_data["config_m"] == source_m) &
                (within_config_data["n_train"] == n_train)
            ]
            
            if baseline_group.empty:
                continue
            
            baseline_accuracies = baseline_group["accuracy"].values
            
            transfer_tests = {}
            
            for (transfer_condition, target_L, target_m), transfer_group in source_group.groupby([
                "transfer_condition", "target_config_L", "target_config_m"
            ]):
                target_key = f"{transfer_condition}_L{target_L}_m{target_m}"
                
                transfer_accuracies = transfer_group["accuracy"].values
                
                if len(transfer_accuracies) < 3 or len(baseline_accuracies) < 3:
                    continue
                
                # Perform statistical test
                if self.config.transfer_significance_test == "ttest":
                    # Use independent t-test
                    statistic, p_value = stats.ttest_ind(baseline_accuracies, transfer_accuracies)
                elif self.config.transfer_significance_test == "wilcoxon":
                    # Use Wilcoxon rank-sum test
                    statistic, p_value = stats.mannwhitneyu(
                        baseline_accuracies, transfer_accuracies, alternative="two-sided"
                    )
                else:  # permutation test
                    statistic, p_value = bootstrap_test(
                        baseline_accuracies, transfer_accuracies, 
                        test_statistic=lambda x, y: np.mean(x) - np.mean(y),
                        n_bootstrap=1000
                    )
                
                effect_size = (np.mean(baseline_accuracies) - np.mean(transfer_accuracies)) / np.sqrt(
                    (np.var(baseline_accuracies) + np.var(transfer_accuracies)) / 2
                )
                
                transfer_tests[target_key] = {
                    "transfer_condition": transfer_condition,
                    "target_config_L": target_L,
                    "target_config_m": target_m,
                    "test_statistic": float(statistic),
                    "p_value": float(p_value),
                    "effect_size": float(effect_size),
                    "baseline_mean": float(np.mean(baseline_accuracies)),
                    "transfer_mean": float(np.mean(transfer_accuracies)),
                    "baseline_n": len(baseline_accuracies),
                    "transfer_n": len(transfer_accuracies)
                }
            
            if transfer_tests:
                # Apply multiple comparisons correction
                p_values = [test["p_value"] for test in transfer_tests.values()]
                test_names = list(transfer_tests.keys())
                
                if self.config.multiple_comparisons_correction != "none":
                    if self.config.multiple_comparisons_correction == "bonferroni":
                        method = "bonferroni"
                    else:  # fdr
                        method = "fdr_bh"
                    
                    rejected, p_corrected, _, _ = multipletests(p_values, method=method)
                    
                    for i, test_name in enumerate(test_names):
                        transfer_tests[test_name]["p_corrected"] = float(p_corrected[i])
                        transfer_tests[test_name]["significant"] = bool(rejected[i])
                else:
                    for test_name in test_names:
                        transfer_tests[test_name]["p_corrected"] = transfer_tests[test_name]["p_value"]
                        transfer_tests[test_name]["significant"] = transfer_tests[test_name]["p_value"] < 0.05
                
                significance_results[source_key] = {
                    "source_config_L": source_L,
                    "source_config_m": source_m,
                    "source_n_train": n_train,
                    "transfer_tests": transfer_tests,
                    "n_tests": len(transfer_tests)
                }
        
        return significance_results
    
    def _analyze_configuration_similarity(self, transfer_degradation: dict[str, t.Any]) -> dict[str, t.Any]:
        """Analyze how configuration similarity affects transfer performance."""
        similarity_analysis = {}
        
        # Extract all configuration pairs and their transfer performance
        config_pairs = []
        
        for source_key, source_data in transfer_degradation.items():
            source_L = source_data["source_config_L"]
            source_m = source_data["source_config_m"]
            
            for target_key, target_data in source_data["transfer_results"].items():
                target_L = target_data["target_config_L"]
                target_m = target_data["target_config_m"]
                degradation = target_data["mean_degradation"]
                
                # Compute various similarity metrics
                l_similarity = 1.0 / (1.0 + abs(source_L - target_L))
                m_similarity = 1.0 / (1.0 + abs(source_m - target_m))
                combined_similarity = (l_similarity + m_similarity) / 2.0
                
                config_pairs.append({
                    "source_config": (source_L, source_m),
                    "target_config": (target_L, target_m),
                    "l_similarity": l_similarity,
                    "m_similarity": m_similarity,
                    "combined_similarity": combined_similarity,
                    "degradation": degradation,
                    "transfer_condition": target_data["transfer_condition"]
                })
        
        if not config_pairs:
            return similarity_analysis
        
        pairs_df = pd.DataFrame(config_pairs)
        
        # Compute correlations between similarity and performance
        similarity_metrics = ["l_similarity", "m_similarity", "combined_similarity"]
        
        for metric in similarity_metrics:
            correlation, p_value = stats.spearmanr(pairs_df[metric], -pairs_df["degradation"])  # Negative because less degradation is better
            
            similarity_analysis[metric] = {
                "correlation": float(correlation),
                "p_value": float(p_value),
                "significant": p_value < 0.05
            }
        
        # Analyze by transfer condition
        condition_similarities = {}
        for condition in pairs_df["transfer_condition"].unique():
            condition_data = pairs_df[pairs_df["transfer_condition"] == condition]
            
            condition_correlations = {}
            for metric in similarity_metrics:
                correlation, p_value = stats.spearmanr(condition_data[metric], -condition_data["degradation"])
                condition_correlations[metric] = {
                    "correlation": float(correlation),
                    "p_value": float(p_value),
                    "significant": p_value < 0.05,
                    "n_samples": len(condition_data)
                }
            
            condition_similarities[condition] = condition_correlations
        
        similarity_analysis["by_condition"] = condition_similarities
        
        return similarity_analysis
    
    def _generate_transfer_visualizations(self, results: dict[str, t.Any]) -> None:
        """Generate comprehensive transfer analysis visualizations."""
        setup_publication_style()
        
        output_dir = self.output_dir / "visualizations"
        output_dir.mkdir(exist_ok=True)
        
        # 1. Transfer matrices heatmaps
        if "transfer_matrices" in results:
            self._plot_transfer_matrices(results["transfer_matrices"], output_dir)
        
        # 2. Degradation by transfer condition
        if "transfer_patterns" in results:
            self._plot_transfer_patterns(results["transfer_patterns"], output_dir)
        
        # 3. Configuration distance effects
        if "transfer_degradation" in results:
            self._plot_distance_effects(results["transfer_degradation"], output_dir)
        
        # 4. Significance analysis
        if "significance_analysis" in results:
            self._plot_significance_analysis(results["significance_analysis"], output_dir)
        
        # 5. Similarity analysis
        if "similarity_analysis" in results:
            self._plot_similarity_analysis(results["similarity_analysis"], output_dir)
    
    def _plot_transfer_matrices(self, transfer_matrices: dict[str, t.Any], output_dir: Path) -> None:
        """Plot transfer performance matrices."""
        for n_train_key, matrix_data in transfer_matrices.items():
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
            
            # Transfer performance matrix
            transfer_matrix = np.array(matrix_data["transfer_matrix"])
            labels = matrix_data["config_labels"]
            
            im1 = ax1.imshow(transfer_matrix, cmap=self.config.matrix_colormap, aspect="auto")
            ax1.set_xticks(range(len(labels)))
            ax1.set_yticks(range(len(labels)))
            ax1.set_xticklabels(labels, rotation=45, ha="right")
            ax1.set_yticklabels(labels)
            ax1.set_title(f"Transfer Performance Matrix ({n_train_key})")
            ax1.set_xlabel("Target Configuration")
            ax1.set_ylabel("Source Configuration")
            
            # Add colorbar
            cbar1 = plt.colorbar(im1, ax=ax1)
            cbar1.set_label("ICL Accuracy")
            
            # Degradation matrix
            degradation_matrix = np.array(matrix_data["degradation_matrix"])
            
            im2 = ax2.imshow(degradation_matrix, cmap=self.config.degradation_colormap, aspect="auto")
            ax2.set_xticks(range(len(labels)))
            ax2.set_yticks(range(len(labels)))
            ax2.set_xticklabels(labels, rotation=45, ha="right")
            ax2.set_yticklabels(labels)
            ax2.set_title(f"Transfer Degradation Matrix ({n_train_key})")
            ax2.set_xlabel("Target Configuration")
            ax2.set_ylabel("Source Configuration")
            
            # Add colorbar
            cbar2 = plt.colorbar(im2, ax=ax2)
            cbar2.set_label("Relative Performance Degradation")
            
            plt.tight_layout()
            save_figure(fig, output_dir / f"transfer_matrices_{n_train_key}.png")
            plt.close()
    
    def _plot_transfer_patterns(self, transfer_patterns: dict[str, t.Any], output_dir: Path) -> None:
        """Plot transfer degradation patterns by condition."""
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. Degradation by transfer condition
        condition_data = transfer_patterns["condition_analysis"]
        if condition_data:
            conditions = list(condition_data.keys())
            means = [condition_data[cond]["mean_degradation"] for cond in conditions]
            stds = [condition_data[cond]["std_degradation"] for cond in conditions]
            
            bars = ax1.bar(conditions, means, yerr=stds, capsize=5, alpha=0.7)
            ax1.set_ylabel("Mean Degradation")
            ax1.set_title("Transfer Degradation by Condition")
            ax1.axhline(y=self.config.degradation_threshold, color='red', linestyle='--', 
                       label=f'Threshold ({self.config.degradation_threshold})')
            ax1.legend()
            
            # Add value labels on bars
            for bar, mean in zip(bars, means):
                height = bar.get_height()
                ax1.text(bar.get_x() + bar.get_width()/2., height + max(stds)/20,
                        f'{mean:.3f}', ha='center', va='bottom')
        
        # 2. Hierarchy (L) distance effects
        hierarchy_data = transfer_patterns["hierarchy_effects"]
        if hierarchy_data:
            distances = sorted([int(d) for d in hierarchy_data.keys()])
            means = [hierarchy_data[str(d)]["mean_degradation"] for d in distances]
            stds = [hierarchy_data[str(d)]["std_degradation"] for d in distances]
            
            ax2.errorbar(distances, means, yerr=stds, marker='o', capsize=5)
            ax2.set_xlabel("Hierarchy Distance (|L_source - L_target|)")
            ax2.set_ylabel("Mean Degradation")
            ax2.set_title("Degradation vs Hierarchy Distance")
            ax2.grid(True, alpha=0.3)
        
        # 3. Diversity (m) distance effects
        diversity_data = transfer_patterns["diversity_effects"]
        if diversity_data:
            distances = sorted([int(d) for d in diversity_data.keys()])
            means = [diversity_data[str(d)]["mean_degradation"] for d in distances]
            stds = [diversity_data[str(d)]["std_degradation"] for d in distances]
            
            ax3.errorbar(distances, means, yerr=stds, marker='s', capsize=5, color='orange')
            ax3.set_xlabel("Diversity Distance (|m_source - m_target|)")
            ax3.set_ylabel("Mean Degradation")
            ax3.set_title("Degradation vs Diversity Distance")
            ax3.grid(True, alpha=0.3)
        
        # 4. Total distance effects
        distance_data = transfer_patterns["distance_effects"]
        if distance_data:
            distances = sorted([int(d) for d in distance_data.keys()])
            means = [distance_data[str(d)]["mean_degradation"] for d in distances]
            stds = [distance_data[str(d)]["std_degradation"] for d in distances]
            
            ax4.errorbar(distances, means, yerr=stds, marker='^', capsize=5, color='green')
            ax4.set_xlabel("Total Distance (|L_source - L_target| + |m_source - m_target|)")
            ax4.set_ylabel("Mean Degradation")
            ax4.set_title("Degradation vs Total Configuration Distance")
            ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        save_figure(fig, output_dir / "transfer_patterns.png")
        plt.close()
    
    def _plot_distance_effects(self, transfer_degradation: dict[str, t.Any], output_dir: Path) -> None:
        """Plot detailed distance effects on transfer performance."""
        # Collect all transfer data
        all_transfers = []
        
        for source_key, source_data in transfer_degradation.items():
            source_L = source_data["source_config_L"]
            source_m = source_data["source_config_m"]
            source_n = source_data["source_n_train"]
            
            for target_key, target_data in source_data["transfer_results"].items():
                target_L = target_data["target_config_L"]
                target_m = target_data["target_config_m"]
                degradation = target_data["mean_degradation"]
                condition = target_data["transfer_condition"]
                
                l_distance = abs(source_L - target_L)
                m_distance = abs(source_m - target_m)
                
                all_transfers.append({
                    "l_distance": l_distance,
                    "m_distance": m_distance,
                    "total_distance": l_distance + m_distance,
                    "degradation": degradation,
                    "condition": condition,
                    "source_n_train": source_n
                })
        
        if not all_transfers:
            return
        
        df = pd.DataFrame(all_transfers)
        
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. Scatter plot: L distance vs degradation
        for condition in df["condition"].unique():
            condition_data = df[df["condition"] == condition]
            ax1.scatter(condition_data["l_distance"], condition_data["degradation"], 
                       label=condition, alpha=0.6)
        
        ax1.set_xlabel("Hierarchy Distance (L)")
        ax1.set_ylabel("Performance Degradation")
        ax1.set_title("Degradation vs Hierarchy Distance")
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # 2. Scatter plot: m distance vs degradation
        for condition in df["condition"].unique():
            condition_data = df[df["condition"] == condition]
            ax2.scatter(condition_data["m_distance"], condition_data["degradation"], 
                       label=condition, alpha=0.6)
        
        ax2.set_xlabel("Diversity Distance (m)")
        ax2.set_ylabel("Performance Degradation")
        ax2.set_title("Degradation vs Diversity Distance")
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # 3. Heatmap: L vs m distance
        pivot_data = df.groupby(["l_distance", "m_distance"])["degradation"].mean().unstack(fill_value=np.nan)
        
        if not pivot_data.empty:
            im = ax3.imshow(pivot_data.values, cmap="Reds", aspect="auto")
            ax3.set_xticks(range(len(pivot_data.columns)))
            ax3.set_yticks(range(len(pivot_data.index)))
            ax3.set_xticklabels(pivot_data.columns)
            ax3.set_yticklabels(pivot_data.index)
            ax3.set_xlabel("Diversity Distance (m)")
            ax3.set_ylabel("Hierarchy Distance (L)")
            ax3.set_title("Mean Degradation Heatmap")
            
            # Add colorbar
            cbar = plt.colorbar(im, ax=ax3)
            cbar.set_label("Mean Degradation")
        
        # 4. Box plot by n_train
        df.boxplot(column="degradation", by="source_n_train", ax=ax4)
        ax4.set_xlabel("Source Training Diversity (n_train)")
        ax4.set_ylabel("Performance Degradation")
        ax4.set_title("Degradation Distribution by Training Diversity")
        plt.suptitle("")  # Remove default title
        
        plt.tight_layout()
        save_figure(fig, output_dir / "distance_effects_detailed.png")
        plt.close()
    
    def _plot_significance_analysis(self, significance_analysis: dict[str, t.Any], output_dir: Path) -> None:
        """Plot statistical significance analysis results."""
        # Collect significance data
        all_tests = []
        
        for source_key, source_data in significance_analysis.items():
            for target_key, test_data in source_data["transfer_tests"].items():
                all_tests.append({
                    "source_config": f"L{source_data['source_config_L']}_m{source_data['source_config_m']}",
                    "target_config": f"L{test_data['target_config_L']}_m{test_data['target_config_M']}",
                    "condition": test_data["transfer_condition"],
                    "p_value": test_data["p_value"],
                    "p_corrected": test_data["p_corrected"],
                    "effect_size": test_data["effect_size"],
                    "significant": test_data["significant"],
                    "baseline_mean": test_data["baseline_mean"],
                    "transfer_mean": test_data["transfer_mean"]
                })
        
        if not all_tests:
            return
        
        df = pd.DataFrame(all_tests)
        
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. P-value distribution
        ax1.hist(df["p_value"], bins=20, alpha=0.7, edgecolor='black')
        ax1.axvline(x=0.05, color='red', linestyle='--', label='α = 0.05')
        ax1.set_xlabel("P-value")
        ax1.set_ylabel("Frequency")
        ax1.set_title("Distribution of P-values")
        ax1.legend()
        
        # 2. Effect size distribution
        ax2.hist(df["effect_size"], bins=20, alpha=0.7, edgecolor='black', color='orange')
        ax2.set_xlabel("Effect Size (Cohen's d)")
        ax2.set_ylabel("Frequency")
        ax2.set_title("Distribution of Effect Sizes")
        ax2.axvline(x=0, color='black', linestyle='-', alpha=0.5)
        
        # 3. Significance by condition
        condition_counts = df.groupby(["condition", "significant"]).size().unstack(fill_value=0)
        
        if not condition_counts.empty:
            condition_counts.plot(kind="bar", ax=ax3, stacked=True)
            ax3.set_xlabel("Transfer Condition")
            ax3.set_ylabel("Number of Tests")
            ax3.set_title("Significance Results by Condition")
            ax3.legend(["Not Significant", "Significant"])
            ax3.tick_params(axis='x', rotation=45)
        
        # 4. Effect size vs p-value
        colors = ['red' if sig else 'blue' for sig in df["significant"]]
        ax4.scatter(df["effect_size"], -np.log10(df["p_value"]), c=colors, alpha=0.6)
        ax4.axhline(y=-np.log10(0.05), color='red', linestyle='--', label='α = 0.05')
        ax4.set_xlabel("Effect Size")
        ax4.set_ylabel("-log10(p-value)")
        ax4.set_title("Volcano Plot: Effect Size vs Significance")
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        save_figure(fig, output_dir / "significance_analysis.png")
        plt.close()
    
    def _plot_similarity_analysis(self, similarity_analysis: dict[str, t.Any], output_dir: Path) -> None:
        """Plot configuration similarity analysis results."""
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. Overall similarity correlations
        metrics = ["l_similarity", "m_similarity", "combined_similarity"]
        correlations = [similarity_analysis[metric]["correlation"] for metric in metrics]
        p_values = [similarity_analysis[metric]["p_value"] for metric in metrics]
        
        bars = ax1.bar(metrics, correlations, 
                      color=['red' if p < 0.05 else 'gray' for p in p_values])
        ax1.set_ylabel("Correlation with Transfer Success")
        ax1.set_title("Similarity Metrics vs Transfer Performance")
        ax1.tick_params(axis='x', rotation=45)
        
        # Add significance indicators
        for i, (bar, p_val) in enumerate(zip(bars, p_values)):
            height = bar.get_height()
            if p_val < 0.05:
                ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                        f'p={p_val:.3f}*', ha='center', va='bottom')
            else:
                ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                        f'p={p_val:.3f}', ha='center', va='bottom')
        
        # 2. Correlation by transfer condition
        if "by_condition" in similarity_analysis:
            condition_data = similarity_analysis["by_condition"]
            conditions = list(condition_data.keys())
            
            for i, metric in enumerate(["l_similarity", "m_similarity", "combined_similarity"]):
                metric_corrs = [condition_data[cond][metric]["correlation"] for cond in conditions]
                metric_ps = [condition_data[cond][metric]["p_value"] for cond in conditions]
                
                x_pos = np.arange(len(conditions)) + i * 0.25
                bars = ax2.bar(x_pos, metric_corrs, width=0.25, label=metric.replace('_', ' ').title(),
                              alpha=0.7)
                
                # Add significance markers
                for bar, p_val in zip(bars, metric_ps):
                    if p_val < 0.05:
                        height = bar.get_height()
                        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                                '*', ha='center', va='bottom', fontweight='bold')
            
            ax2.set_xlabel("Transfer Condition")
            ax2.set_ylabel("Correlation")
            ax2.set_title("Similarity Correlations by Condition")
            ax2.set_xticks(np.arange(len(conditions)) + 0.25)
            ax2.set_xticklabels(conditions)
            ax2.legend()
        
        # 3. & 4. Placeholder for additional similarity analyses
        ax3.text(0.5, 0.5, "Additional similarity\nanalysis plots\ncan be added here",
                ha='center', va='center', transform=ax3.transAxes, fontsize=12)
        ax3.set_title("Future Similarity Analysis")
        
        ax4.text(0.5, 0.5, "Configuration space\nvisualization\ncan be added here",
                ha='center', va='center', transform=ax4.transAxes, fontsize=12)
        ax4.set_title("Configuration Space Visualization")
        
        plt.tight_layout()
        save_figure(fig, output_dir / "similarity_analysis.png")
        plt.close()


def run_rq4_analysis(
    data_dir: Path,
    output_dir: Path,
    config: TransferConfig | None = None
) -> dict[str, t.Any]:
    """Run RQ4 transfer analysis with default configuration."""
    if config is None:
        config = TransferConfig(
            data_dir=data_dir,
            output_dir=output_dir,
            transfer_conditions=["cross_L", "cross_m", "cross_config"],
            min_samples_for_transfer=5,
            degradation_threshold=0.1,
            compute_transfer_matrices=True,
            matrix_aggregation="mean",
            transfer_significance_test="ttest",
            multiple_comparisons_correction="bonferroni"
        )
    
    analyzer = TransferAnalyzer(config)
    return analyzer.run_analysis()


# Test function
def test_transfer_analyzer():
    """Test the transfer analyzer with sample data."""
    # This would typically use real data paths
    data_dir = Path("./test_data")
    output_dir = Path("./test_output/rq4")
    
    # Create test configuration
    config = TransferConfig(
        data_dir=data_dir,
        output_dir=output_dir,
        transfer_conditions=["cross_L", "cross_m"],
        min_samples_for_transfer=3,
        degradation_threshold=0.05,
        matrix_aggregation="mean"
    )
    
    try:
        results = run_rq4_analysis(data_dir, output_dir, config)
        print("✓ RQ4 Transfer analysis completed successfully")
        print(f"✓ Results saved to {output_dir}")
        return results
    except Exception as e:
        print(f"✗ Transfer analysis failed: {e}")
        return None


if __name__ == "__main__":
    test_transfer_analyzer()

# Q5 Diversity analysis

In [ ]:
"""RQ5: Analyze task diversity effects on out-of-distribution performance."""

from pathlib import Path
from dataclasses import dataclass, field
import typing as t
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from .base_analyzer import BaseAnalyzer, AnalysisConfig
from ..utils.data_loaders import create_filter
from ..utils.statistical_utils import compute_confidence_interval, bootstrap_test, fit_power_law
from ..utils.visualization_utils import setup_publication_style, save_figure


@dataclass
class DiversityConfig(AnalysisConfig):
    """Configuration for diversity analysis."""
    # Diversity analysis parameters
    min_n_train_samples: int = 3
    max_n_train_for_analysis: int | None = None
    
    # OOD vs ID comparison
    ood_transfer_conditions: list[str] = field(default_factory=lambda: [
        "cross_L", "cross_m", "cross_config"
    ])
    id_transfer_condition: str = "within_config"
    
    # Memorization vs generalization analysis
    analyze_memorization: bool = True
    context_size_thresholds: list[int] = field(default_factory=lambda: [1, 5, 10, 20])
    
    # Scaling law fitting
    fit_scaling_laws: bool = True
    scaling_law_forms: list[str] = field(default_factory=lambda: [
        "power", "exponential", "logarithmic"
    ])
    
    # Statistical analysis
    diversity_significance_test: str = "anova"  # "anova", "kruskal"
    correlation_method: str = "spearman"  # "pearson", "spearman"
    
    # Visualization parameters
    diversity_colormap: str = "viridis"
    performance_colormap: str = "RdYlBu_r"


class DiversityAnalyzer(BaseAnalyzer):
    """Analyzer for task diversity effects on OOD performance (RQ5)."""
    
    def __init__(self, config: DiversityConfig):
        """Initialize diversity analyzer."""
        super().__init__(config)
        self.config: DiversityConfig = config
    
    def run_analysis(self) -> dict[str, t.Any]:
        """Run comprehensive diversity analysis."""
        print("Starting RQ5: Diversity Analysis")
        print("=" * 50)
        
        # Load and validate data
        id_data = self._load_id_performance_data()
        ood_data = self._load_ood_performance_data()
        
        if id_data.empty:
            raise ValueError("Insufficient in-distribution data for diversity analysis")
        
        print(f"Loaded {len(id_data)} in-distribution evaluation records")
        print(f"Loaded {len(ood_data)} out-of-distribution evaluation records")
        print(f"n_train levels: {sorted(id_data['n_train'].unique())}")
        
        results = {}
        
        # 1. Analyze diversity scaling on ID performance
        print("\n1. Analyzing diversity scaling on ID performance...")
        id_scaling = self._analyze_id_diversity_scaling(id_data)
        results["id_diversity_scaling"] = id_scaling
        
        # 2. Compare ID vs OOD performance scaling
        print("2. Comparing ID vs OOD performance scaling...")
        if not ood_data.empty:
            ood_comparison = self._compare_id_ood_scaling(id_data, ood_data)
            results["id_ood_comparison"] = ood_comparison
        
        # 3. Analyze memorization vs generalization trade-offs
        print("3. Analyzing memorization vs generalization...")
        if self.config.analyze_memorization:
            memorization_analysis = self._analyze_memorization_generalization(id_data, ood_data)
            results["memorization_analysis"] = memorization_analysis
        
        # 4. Fit diversity scaling laws
        print("4. Fitting diversity scaling laws...")
        if self.config.fit_scaling_laws:
            scaling_laws = self._fit_diversity_scaling_laws(id_data, ood_data)
            results["scaling_laws"] = scaling_laws
        
        # 5. Analyze diversity-context interactions
        print("5. Analyzing diversity-context interactions...")
        interaction_analysis = self._analyze_diversity_context_interactions(id_data, ood_data)
        results["diversity_context_interactions"] = interaction_analysis
        
        # 6. Statistical significance testing
        print("6. Testing diversity effects significance...")
        significance_analysis = self._test_diversity_significance(id_data, ood_data)
        results["significance_analysis"] = significance_analysis
        
        # 7. Generate visualizations
        print("7. Generating visualizations...")
        self._generate_diversity_visualizations(results)
        
        # Save results
        self.save_results(results, "rq5_diversity_results.json")
        
        print("\nRQ5 Analysis completed successfully!")
        return results
    
    def _load_id_performance_data(self) -> pd.DataFrame:
        """Load in-distribution performance data."""
        filter_dict = (create_filter()
                      .transfer_condition(self.config.id_transfer_condition)
                      .control_type("normal")
                      .build())
        
        data = self.data_loader.load_icl_performance(filters=filter_dict)
        
        # Add model metadata
        data = data.merge(
            self.model_registry[["model_id", "config_L", "config_m", "n_train", "checkpoint_step"]],
            on="model_id",
            how="left"
        )
        
        # Filter by n_train if specified
        if self.config.max_n_train_for_analysis is not None:
            data = data[data["n_train"] <= self.config.max_n_train_for_analysis]
        
        return data
    
    def _load_ood_performance_data(self) -> pd.DataFrame:
        """Load out-of-distribution performance data."""
        filter_dict = (create_filter()
                      .transfer_condition(self.config.ood_transfer_conditions)
                      .control_type("normal")
                      .build())
        
        data = self.data_loader.load_icl_performance(filters=filter_dict)
        
        # Add model metadata
        data = data.merge(
            self.model_registry[["model_id", "config_L", "config_m", "n_train", "checkpoint_step"]],
            on="model_id",
            how="left"
        )
        
        # Filter by n_train if specified
        if self.config.max_n_train_for_analysis is not None:
            data = data[data["n_train"] <= self.config.max_n_train_for_analysis]
        
        return data
    
    def _analyze_id_diversity_scaling(self, id_data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze how task diversity affects in-distribution performance."""
        scaling_results = {}
        
        # Group by configuration and analyze diversity scaling
        for (config_L, config_m), config_group in id_data.groupby(["config_L", "config_m"]):
            config_key = f"L{config_L}_m{config_m}"
            
            # Get performance vs diversity data
            diversity_performance = []
            
            for n_train in sorted(config_group["n_train"].unique()):
                n_train_data = config_group[config_group["n_train"] == n_train]
                
                if len(n_train_data) < self.config.min_n_train_samples:
                    continue
                
                # Aggregate across context sizes and models
                performance_stats = {
                    "n_train": n_train,
                    "mean_accuracy": float(n_train_data["accuracy"].mean()),
                    "std_accuracy": float(n_train_data["accuracy"].std()),
                    "median_accuracy": float(n_train_data["accuracy"].median()),
                    "max_accuracy": float(n_train_data["accuracy"].max()),
                    "min_accuracy": float(n_train_data["accuracy"].min()),
                    "n_samples": len(n_train_data),
                    "n_models": n_train_data["model_id"].nunique(),
                    "n_contexts": n_train_data["context_size"].nunique()
                }
                
                # Compute confidence intervals
                ci = compute_confidence_interval(n_train_data["accuracy"].values)
                performance_stats["ci_lower"] = float(ci[0])
                performance_stats["ci_upper"] = float(ci[1])
                
                # Analyze by context size
                context_performance = {}
                for context_size in sorted(n_train_data["context_size"].unique()):
                    context_data = n_train_data[n_train_data["context_size"] == context_size]
                    context_performance[int(context_size)] = {
                        "mean_accuracy": float(context_data["accuracy"].mean()),
                        "std_accuracy": float(context_data["accuracy"].std()),
                        "n_samples": len(context_data)
                    }
                
                performance_stats["by_context"] = context_performance
                diversity_performance.append(performance_stats)
            
            if len(diversity_performance) >= 2:
                # Compute scaling trends
                n_trains = [p["n_train"] for p in diversity_performance]
                accuracies = [p["mean_accuracy"] for p in diversity_performance]
                
                # Linear correlation
                correlation, p_value = stats.spearmanr(n_trains, accuracies)
                
                # Fit trends
                trends = self._fit_diversity_trends(n_trains, accuracies)
                
                scaling_results[config_key] = {
                    "config_L": config_L,
                    "config_m": config_m,
                    "diversity_performance": diversity_performance,
                    "correlation": float(correlation),
                    "correlation_p_value": float(p_value),
                    "trends": trends,
                    "n_diversity_levels": len(diversity_performance)
                }
        
        return scaling_results
    
    def _compare_id_ood_scaling(self, id_data: pd.DataFrame, ood_data: pd.DataFrame) -> dict[str, t.Any]:
        """Compare in-distribution vs out-of-distribution diversity scaling."""
        comparison_results = {}
        
        # Group by source configuration
        for (config_L, config_m), id_group in id_data.groupby(["config_L", "config_m"]):
            config_key = f"L{config_L}_m{config_m}"
            
            # Get ID scaling
            id_scaling = []
            for n_train in sorted(id_group["n_train"].unique()):
                n_train_data = id_group[id_group["n_train"] == n_train]
                if len(n_train_data) >= self.config.min_n_train_samples:
                    id_scaling.append({
                        "n_train": n_train,
                        "mean_accuracy": float(n_train_data["accuracy"].mean()),
                        "std_accuracy": float(n_train_data["accuracy"].std())
                    })
            
            # Get OOD scaling for each transfer condition
            ood_scaling_by_condition = {}
            
            ood_config_data = ood_data[
                (ood_data["config_L"] == config_L) &
                (ood_data["config_m"] == config_m)
            ]
            
            for transfer_condition in self.config.ood_transfer_conditions:
                condition_data = ood_config_data[
                    ood_config_data["transfer_condition"] == transfer_condition
                ]
                
                if condition_data.empty:
                    continue

# Q6 Comparative Analysis